# 📦 Virtual Environments & Packaging — Ship Python Code That Runs Anywhere

> **What you'll learn:** why environments exist, how `venv`, `pip`, lock files and **uv** work, how imports find modules, and how to turn your code into a real package with `pyproject.toml`, a wheel, tests, a CLI command, and reproducible, secret-free configuration. Every command is actually run from this notebook in a temporary folder.

| | |
|---|---|
| **Difficulty** | 🟢 Beginner → 🟡 Intermediate (🔴 reproducible ML environments) |
| **Time** | ~3 hours to read and run, +1.5 hours for exercises and the project |
| **Prerequisites** | [OOP in Python](03_OOP_in_Python.ipynb) (classes, modules you've written) |
| **Tested with** | Python 3.12 · uv 0.10 · setuptools 84 · build 1.6 · pytest 9 · packaging 26 |
| **Interview relevance** | ⭐⭐ Medium–High — "works on my machine" debugging, lock files, `pyproject.toml`, wheels, CUDA wheels, secrets hygiene |

## 🤔 What Is an Environment (and a Package)?

Imagine a restaurant with one shared kitchen. The pastry chef needs *salted* butter, the sauce chef needs *unsalted* — but there's only one butter shelf. Somebody's recipe breaks. The fix: **give each chef their own kitchen**.

- A **Python environment** is a kitchen: one Python interpreter plus the set of installed libraries it can import.
- A **virtual environment** ("venv") is a lightweight private kitchen for *one project*, so projects can't break each other.
- A **package** is a box of code you can install, with a label (**metadata**: name, version, dependencies).
- A **dependency** is another package your code needs. Its own dependencies are **transitive dependencies**.
- **PyPI** (the Python Package Index, pypi.org) is the public warehouse that `pip install` and `uv add` download from.
- A **wheel** (`.whl`) is a ready-to-install package file; an **sdist** (`.tar.gz`) is a source bundle that must be built first.

```
~/projects/fraud-model/.venv   →  Python 3.12 + numpy 1.26 + scikit-learn 1.4
~/projects/llm-chatbot/.venv   →  Python 3.12 + numpy 2.2  + torch 2.5
```

## 🎯 Why It Matters

- **"It works on my machine"** is the most common ML engineering bug. Different library versions change results, break imports, or silently install CPU-only PyTorch on a GPU server.
- **Every deployment needs it:** Docker images, CI pipelines, model-serving containers, and Airflow workers all start from a dependency file and a lock file.
- **Sharing code** — an internal feature-engineering library, an evaluation toolkit, a CLI — means packaging it with `pyproject.toml` so teammates can `pip install` it.
- **In interviews** you'll get debugging questions ("pip install worked but import fails in Jupyter", "CI broke after a release", "`torch.cuda.is_available()` is False") and design questions ("how do you make training reproducible?", "how do you manage secrets?"). This notebook covers them with real commands.

## ✅ By the End You Can

- [ ] Explain what a virtual environment physically is and create one with `venv` or `uv`
- [ ] Read and write version specifiers, and explain requirements files vs lock files
- [ ] Use uv's project workflow: `uv init`, `uv add`, `uv lock`, `uv sync`, `uv run`
- [ ] Explain how imports work, including namespace packages (PEP 420)
- [ ] Package a project with a `src/` layout and `pyproject.toml`, build a wheel, install it, and run its CLI and tests
- [ ] Make ML environments reproducible (Python, lock file, CUDA wheels) and keep secrets out of code

## 📋 Table of Contents

1. [Why Environments Exist](#1.-Why-Environments-Exist-🟢)
2. [venv and pip](#2.-venv-and-pip-🟢)
3. [Version Specifiers](#3.-Version-Specifiers-🟢)
4. [requirements.txt vs Lock Files](#4.-requirements.txt-vs-Lock-Files-🟡)
5. [uv — The Modern Default](#5.-uv-—-The-Modern-Default-🟡)
6. [pip-tools, Poetry, and conda](#6.-pip-tools,-Poetry,-and-conda-🟢)
7. [Modules, Packages, and Namespace Packages](#7.-Modules,-Packages,-and-Namespace-Packages-🟡)
8. [Project Layout and pyproject.toml](#8.-Project-Layout-and-pyproject.toml-🟡)
9. [Building and Installing — Wheels, sdists, Editable Installs](#9.-Building-and-Installing-—-Wheels,-sdists,-Editable-Installs-🟡)
10. [Testing with pytest](#10.-Testing-with-pytest-🟢)
11. [Linting and Formatting with Ruff](#11.-Linting-and-Formatting-with-Ruff-🟢)
12. [Reproducible ML Environments](#12.-Reproducible-ML-Environments-🔴)
13. [.gitignore and Secrets](#13.-.gitignore-and-Secrets-🟢)
- [🔧 Build It From Scratch](#🔧-Build-It-From-Scratch) · [⚠️ Common Pitfalls](#⚠️-Common-Pitfalls) · [🏋️ Practice Exercises](#🏋️-Practice-Exercises) · [🚀 Mini Project](#🚀-Mini-Project:-Package-a-Text-Statistics-CLI) · [🎤 Interview Q&A](#🎤-Interview-Q&A) · [🧪 Quick Quiz](#🧪-Quick-Quiz) · [📚 Resources](#📚-Resources) · [📝 Summary](#📝-Summary-Cheat-Sheet)

## ⚙️ Setup

This notebook runs **real terminal commands** (`python -m venv`, `uv`, `python -m build`, `pytest`) through a small `run()` helper that prints `$ command` and its output, just like a terminal. Everything is created inside a **temporary folder** (shown as `$WORK`), so nothing touches your projects. Your home folder is shown as `~`.

**uv** is optional but recommended: install it by following [Installation | uv](https://docs.astral.sh/uv/getting-started/installation/) (e.g. `curl -LsSf https://astral.sh/uv/install.sh | sh` or `brew install uv`). Cells that need it print an honest skip message when it's missing. A few cells download small packages from PyPI (a few MB, cached after the first run).

In [1]:
# %pip install -q build packaging pytest python-dotenv setuptools

import base64
import hashlib
import importlib
import importlib.metadata as md
import importlib.util
import io
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import textwrap
import time
import tomllib
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

from packaging.markers import Marker
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.version import Version

UV = shutil.which("uv")
WORK_RAW = tempfile.mkdtemp(prefix="packaging_nb_")
WORK = Path(WORK_RAW).resolve()
HOME = str(Path.home())
OUTPUT_DIR = Path("_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def short(text):
    """Make paths readable: temp folder → $WORK, home folder → ~."""
    text = str(text)
    for work in {str(WORK), WORK_RAW, "/private" + WORK_RAW}:
        text = text.replace(work, "$WORK")
    return text.replace(HOME, "~")


def clean_env(**extra):
    """A copy of os.environ without variables that would leak the notebook's own environment into commands."""
    env = {k: v for k, v in os.environ.items() if k not in {"VIRTUAL_ENV", "PYTHONPATH", "PYTHONHOME", "CONDA_PREFIX", "UV_PYTHON"}}
    env.update(NO_COLOR="1", UV_NO_PROGRESS="1", PIP_DISABLE_PIP_VERSION_CHECK="1")
    env.update({k: str(v) for k, v in extra.items()})
    return env


def run(cmd, cwd=None, env=None, check=True, show=True, max_lines=20, tail=False, timeout=600):
    """Run a command like a terminal: print `$ command`, its (trimmed) output, and the exit code if non-zero."""
    cmd = [str(c) for c in cmd]
    result = subprocess.run(cmd, cwd=cwd, env=env or clean_env(), capture_output=True, text=True, timeout=timeout)
    if show:
        program = short(cmd[0]) if cmd[0].startswith((str(WORK), WORK_RAW)) else Path(cmd[0]).name
        print("$", short(" ".join([program, *cmd[1:]])))
        lines = (result.stdout + result.stderr).rstrip().splitlines()
        hidden = len(lines) - max_lines
        if hidden > 0:
            lines = lines[-max_lines:] if tail else lines[:max_lines]
            if tail:
                print(f"  … ({hidden} earlier lines hidden)")
        for line in lines:
            print("  " + short(re.sub(r"\x1b\[[0-9;]*m", "", line)))     # drop terminal colour codes
        if hidden > 0 and not tail:
            print(f"  … ({hidden} more lines)")
        if result.returncode != 0:
            print(f"  [exit code {result.returncode}]")
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed ({result.returncode}): {short(' '.join(cmd))}\n{short(result.stderr[-1500:])}")
    return result


def write(path, text):
    """Create a file (and its folders) from an indented triple-quoted string."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"))
    return path


def tree(root):
    """Print a folder tree, skipping generated clutter."""
    root = Path(root)
    skip = {".venv", "__pycache__", ".pytest_cache", "build", "dist", ".ruff_cache"}
    print(f"📁 {root.name}/")
    for p in sorted(root.rglob("*")):
        parts = p.relative_to(root).parts
        if any(part in skip or part.endswith(".egg-info") for part in parts):
            continue
        print("    " * len(parts) + ("📁 " if p.is_dir() else "📄 ") + p.name + ("/" if p.is_dir() else ""))


def venv_bin(venv_dir, name):
    """Path of an executable inside a venv (bin/ on macOS/Linux, Scripts\\ on Windows)."""
    if os.name == "nt":
        return Path(venv_dir) / "Scripts" / f"{name}.exe"
    return Path(venv_dir) / "bin" / name


def make_venv(venv_dir):
    """Create a fresh, empty venv — with uv when available (fast), else the built-in venv module."""
    if UV:
        run([UV, "venv", "--quiet", "--python", sys.executable, venv_dir], show=False)
    else:
        run([sys.executable, "-m", "venv", venv_dir], show=False)
    return venv_bin(venv_dir, "python")


def pip_install(venv_python, *args, check=True):
    """Install into a specific venv (never into the notebook's own environment)."""
    if UV:
        return run([UV, "pip", "install", "--python", venv_python, *args], check=check)
    return run([venv_python, "-m", "pip", "install", *args], check=check)


def dist_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "not installed"


def check(name, got, expected, hint=""):
    """✅ if correct, ⏳ if not attempted yet (None), ❌ AssertionError with a hint otherwise."""
    if got is None or got is ...:
        print(f"⏳ {name}: not attempted yet — replace None with your answer.")
        return
    try:
        ok = bool(got == expected)
    except Exception:
        ok = False
    assert ok, f"❌ {name}: not quite (got {got!r}). {hint}"
    print(f"✅ {name}: correct!")


uv_version = run([UV, "--version"], show=False).stdout.split()[1] if UV else "not found"
print(f"Python {platform.python_version()} | uv {uv_version} | build {dist_version('build')} | "
      f"setuptools {dist_version('setuptools')} | pytest {dist_version('pytest')} | packaging {dist_version('packaging')}")
print(f"scratch folder: a new temporary directory named {WORK.name} (shown as $WORK)")
if not UV:
    print("⚠️ uv not found: uv-specific cells will print a skip message; everything else still runs.")

Python 3.12.11 | uv 0.10.3 | build 1.6.1 | setuptools 84.0.0 | pytest 9.1.1 | packaging 26.3
scratch folder: a new temporary directory named packaging_nb_6g7823sk (shown as $WORK)


## 1. Why Environments Exist 🟢

When Python runs `import numpy`, it searches folders listed in `sys.path`, including one **site-packages** folder where installed libraries live. One interpreter → one site-packages → **one version of each library**.

Let's look at the environment running this notebook:

In [2]:
print("interpreter     :", short(sys.executable))
print("sys.prefix      :", short(sys.prefix))
print("sys.base_prefix :", short(sys.base_prefix))
print("inside a virtual environment?", sys.prefix != sys.base_prefix)

import site

print("libraries install into:", short(site.getsitepackages()[0]))
n_dists = len({d.metadata["Name"].lower() for d in md.distributions()})
print(f"\nthis environment holds {n_dists} installed distributions")
for pkg in ["pandas", "scikit-learn"]:
    reqs = [Requirement(r) for r in (md.requires(pkg) or [])]
    required = sorted({r.name for r in reqs if r.marker is None or r.marker.evaluate({"extra": ""})})
    print(f"{pkg} {md.version(pkg)} requires → {required}")

interpreter     : ~/Workspaces/Learning-AI-ML/.venv/bin/python
sys.prefix      : ~/Workspaces/Learning-AI-ML/.venv
sys.base_prefix : /opt/homebrew/opt/python@3.12/Frameworks/Python.framework/Versions/3.12
inside a virtual environment? True
libraries install into: ~/Workspaces/Learning-AI-ML/.venv/lib/python3.12/site-packages

this environment holds 382 installed distributions
pandas 3.0.5 requires → ['numpy', 'python-dateutil']
scikit-learn 1.9.1 requires → ['joblib', 'narwhals', 'numpy', 'scipy', 'threadpoolctl']


Now the conflict. Suppose an older serving app needs NumPy 1.x and a new training project needs NumPy ≥ 2.1. `SpecifierSet` (from the `packaging` library that pip itself uses) tells us which real NumPy releases each accepts:

In [3]:
numpy_releases = ["1.24.4", "1.26.4", "2.0.2", "2.1.3", "2.2.6"]
serving_app = SpecifierSet("<2.0")
training_project = SpecifierSet(">=2.1")

print("serving app accepts     :", list(serving_app.filter(numpy_releases)))
print("training project accepts:", list(training_project.filter(numpy_releases)))
print("both at once            :", list((serving_app & training_project).filter(numpy_releases)),
      "→ impossible in ONE environment, trivial with two")

serving app accepts     : ['1.24.4', '1.26.4']
training project accepts: ['2.1.3', '2.2.6']
both at once            : [] → impossible in ONE environment, trivial with two


Modern Linux distributions and Homebrew even **block** `pip install` into the system Python (PEP 668, "externally managed environment") to stop you from breaking OS tools. The answer is always the same: one environment per project.

> 💡 **Interview angle:** "Why do we need virtual environments?" — one interpreter can hold only one version of each package; per-project environments isolate conflicting requirements and keep the system Python clean (PEP 668).

## 2. venv and pip 🟢

`venv` ships with Python. A virtual environment is just a **folder**:

```
.venv/
├── pyvenv.cfg          ← "home = <base Python>": tells Python it's a venv
├── bin/  (Scripts\ on Windows)
│   ├── python  →  link to the base interpreter
│   ├── pip
│   └── activate
└── lib/python3.12/site-packages/   ← this project's libraries
```

Terminal workflow (macOS/Linux; Windows uses `.venv\Scripts\activate`):

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install "numpy>=2.0"      # always quote specifiers!
python -m pip list
deactivate
```

Let's actually create one and poke at it:

In [4]:
demo_venv = WORK / "demo-venv"
start = time.perf_counter()
run([sys.executable, "-m", "venv", demo_venv])
print(f"created in {time.perf_counter() - start:.1f} s → contains {sorted(p.name for p in demo_venv.iterdir())}")
print("\npyvenv.cfg:\n" + textwrap.indent(short((demo_venv / "pyvenv.cfg").read_text()), "   "))

demo_python = venv_bin(demo_venv, "python")
run([demo_python, "-c", "import sys; print('sys.prefix =', sys.prefix); print('in a venv:', sys.prefix != sys.base_prefix)"])
run([demo_python, "-m", "pip", "list"])
run([demo_python, "-c", "import numpy"], check=False, tail=True, max_lines=1);

$ python -m venv $WORK/demo-venv
created in 2.3 s → contains ['bin', 'include', 'lib', 'pyvenv.cfg']

pyvenv.cfg:
   home = /opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/bin
   include-system-site-packages = false
   version = 3.12.11
   executable = /opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/bin/python3.12
   command = ~/Workspaces/Learning-AI-ML/.venv/bin/python -m venv $WORK/demo-venv

$ $WORK/demo-venv/bin/python -c import sys; print('sys.prefix =', sys.prefix); print('in a venv:', sys.prefix != sys.base_prefix)
  sys.prefix = $WORK/demo-venv
  in a venv: True


$ $WORK/demo-venv/bin/python -m pip list
  Package Version
  ------- -------
  pip     25.1.1
$ $WORK/demo-venv/bin/python -c import numpy
  … (2 earlier lines hidden)
  ModuleNotFoundError: No module named 'numpy'
  [exit code 1]


NumPy is installed for this notebook, yet the new venv can't see it — that's the isolation working. And **activation is optional**: `activate` only puts the venv's `bin/` first on your `PATH`, so `python` means the venv's Python. Calling `.venv/bin/python` directly (as above) is exactly equivalent — and what Dockerfiles, cron jobs, and CI scripts should do.

In [5]:
activate_script = (demo_venv / "bin" / "activate")
if activate_script.exists():
    key_lines = [line.strip() for line in activate_script.read_text().splitlines()
                 if line.strip().startswith(("export VIRTUAL_ENV=", '_OLD_VIRTUAL_PATH="$PATH"', 'PATH="$VIRTUAL_ENV'))]
    print("the heart of `source .venv/bin/activate` (remember the old PATH, then put the venv's bin/ first):\n   "
          + "\n   ".join(short(line) for line in key_lines))

the heart of `source .venv/bin/activate` (remember the old PATH, then put the venv's bin/ first):
   export VIRTUAL_ENV=$WORK/demo-venv
   _OLD_VIRTUAL_PATH="$PATH"
   PATH="$VIRTUAL_ENV/"bin":$PATH"


### ✍️ Your Turn

Write `venv_python_path(venv_dir, windows)` that returns the path (as a `pathlib.Path`) of the Python interpreter inside a venv: `<venv>/bin/python` normally, `<venv>/Scripts/python.exe` when `windows=True`.

In [6]:
def venv_python_path(venv_dir, windows):
    return None  # TODO: your code here


got = None
if venv_python_path(".venv", windows=False) is not None:
    got = [venv_python_path(".venv", windows=False).as_posix(), venv_python_path(".venv", windows=True).as_posix()]
check("venv_python_path", got, [".venv/bin/python", ".venv/Scripts/python.exe"], hint="Path(venv_dir) / 'bin' / 'python' — and the Windows variant.")

⏳ venv_python_path: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def venv_python_path(venv_dir, windows):
    if windows:
        return Path(venv_dir) / "Scripts" / "python.exe"
    return Path(venv_dir) / "bin" / "python"


got = [venv_python_path(".venv", windows=False).as_posix(), venv_python_path(".venv", windows=True).as_posix()]
check("venv_python_path", got, [".venv/bin/python", ".venv/Scripts/python.exe"])
```
</details>

> 💡 **Interview angle:** "What does `activate` do?" — it only edits `PATH` (and sets `VIRTUAL_ENV`). The venv works because its `python` finds `pyvenv.cfg` and sets `sys.prefix`; that's why `.venv/bin/python script.py` works without activation. Prefer `python -m pip` over bare `pip` so you know which interpreter you're installing into.

## 3. Version Specifiers 🟢

A **version specifier** says which versions of a dependency you accept (PEP 440):

| Specifier | Means | Typical use |
|---|---|---|
| `==2.2.5` | exactly this version | lock files, reproducing a bug |
| `==2.2.*` | any 2.2.x | stay on a minor series |
| `>=2.1,<3` | a range | library dependencies |
| `~=2.2` | "compatible release": `>=2.2, ==2.*` | allow new features, not a new major |
| `~=2.2.0` | `>=2.2.0, ==2.2.*` | allow bug-fix releases only |
| `!=2.2.0` | exclude a known-broken release | |
| `requests[socks]` | also install an **extra** (optional feature set) | |
| `tomli ; python_version < "3.11"` | an **environment marker**: only when true | platform-specific deps |

⚠️ In a terminal, **quote** anything containing `<` or `>`: `pip install numpy>=1.24` is read by the shell as "run `pip install numpy` and redirect output to a file named `=1.24`" (see ⚠️ Pitfall 1).

In [7]:
candidates = ["1.9.0", "2.0.0", "2.1.3", "2.2.0rc1", "2.2.0", "2.2.5", "2.10.1", "3.0.0"]
for spec in ["==2.2.*", ">=2.1,<3", "~=2.2", "~=2.2.0", "!=2.2.0,>=2.2"]:
    print(f"{spec:15s} → {list(SpecifierSet(spec).filter(candidates))}")
print(f"{'>=2.2.0rc1':15s} → {list(SpecifierSet('>=2.2.0rc1').filter(candidates))}   ← pre-releases only when you ask for one")

print("\nversion order:", sorted(["1.10", "1.9", "1.10rc1", "1.10.post1", "1.10.dev0", "1.10a1"], key=Version))
print("string order :", sorted(["1.10", "1.9"]), "← plain string sorting gets versions wrong")

req = Requirement('requests[socks]>=2.31,<3 ; sys_platform == "linux" and python_version >= "3.10"')
print(f"\nname={req.name} extras={req.extras} specifier={req.specifier} marker=({req.marker})")
print("applies on this machine:", req.marker.evaluate(), "| on Linux + Python 3.12:",
      req.marker.evaluate({"sys_platform": "linux", "python_version": "3.12"}))

==2.2.*         → ['2.2.0', '2.2.5']
>=2.1,<3        → ['2.1.3', '2.2.0', '2.2.5', '2.10.1']
~=2.2           → ['2.2.0', '2.2.5', '2.10.1']
~=2.2.0         → ['2.2.0', '2.2.5']
!=2.2.0,>=2.2   → ['2.2.5', '2.10.1', '3.0.0']
>=2.2.0rc1      → ['2.2.0rc1', '2.2.0', '2.2.5', '2.10.1', '3.0.0']   ← pre-releases only when you ask for one

version order: ['1.9', '1.10.dev0', '1.10a1', '1.10rc1', '1.10', '1.10.post1']
string order : ['1.10', '1.9'] ← plain string sorting gets versions wrong

name=requests extras={'socks'} specifier=<3,>=2.31 marker=(sys_platform == "linux" and python_version >= "3.10")
applies on this machine: False | on Linux + Python 3.12: True


### ✍️ Your Turn

Which of `releases` satisfy `~=1.4`? Store them in `compatible` (keep the original order). Predict first, then compute it with `SpecifierSet`.

In [8]:
releases = ["1.3.9", "1.4.0", "1.4.2", "1.9.0", "2.0.0", "1.5.0rc1"]
compatible = None  # TODO
check("compatible", compatible, ["1.4.0", "1.4.2", "1.9.0"],
      hint="~=1.4 means >=1.4 and ==1.*; pre-releases are excluded unless requested.")

⏳ compatible: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
releases = ["1.3.9", "1.4.0", "1.4.2", "1.9.0", "2.0.0", "1.5.0rc1"]
compatible = list(SpecifierSet("~=1.4").filter(releases))
check("compatible", compatible, ["1.4.0", "1.4.2", "1.9.0"])
```
`1.9.0` surprises many people: `~=1.4` allows any 1.x from 1.4 upwards. Use `~=1.4.0` to allow only 1.4.x.
</details>

> 💡 **Interview angle:** "What does `~=2.2` allow?" — `>=2.2, ==2.*`, so 2.9 is fine but 3.0 is not. Also mention that `"1.10" > "1.9"` is `False` as strings, which is why tools parse versions.

## 4. requirements.txt vs Lock Files 🟡

Two different questions:

1. **What does my project need?** — *abstract* dependencies with ranges, e.g. `requests>=2.31`. Lives in `pyproject.toml` (`[project] dependencies`). Right for **libraries**.
2. **Exactly what did we install?** — *concrete*, **locked** dependencies: every package including transitive ones, with exact versions and file hashes. Right for **applications, CI, and model serving**.

| File | What it is |
|---|---|
| hand-written `requirements.txt` | usually a mix of both; no transitive pins |
| `pip freeze > requirements.txt` | a dump of *whatever* is installed (including leftovers); platform-specific |
| `requirements.txt` from `pip-compile` / `uv pip compile` | a real lock for one platform/Python |
| `uv.lock`, `poetry.lock` | tool-specific, cross-platform lock files |
| `pylock.toml` | the **standard** lock file format (PEP 751, 2025) |

Let's lock one direct requirement and see how many packages it really pins:

In [9]:
lock_demo = WORK / "lock-demo"
write(lock_demo / "requirements.in", "requests>=2.31\n")

if not UV:
    print("⏭️ Skipped: uv not installed — `pip install pip-tools && pip-compile requirements.in` demonstrates the same idea.")
else:
    compiled = run([UV, "pip", "compile", "requirements.in", "-o", "requirements.txt", "--python-version", "3.12", "--quiet"],
                   cwd=lock_demo, check=False)
    if compiled.returncode != 0:
        print("⏭️ Skipped: uv could not reach PyPI to resolve versions (offline?).")
    else:
        locked = (lock_demo / "requirements.txt").read_text()
        print(locked)
        pins = [line for line in locked.splitlines() if "==" in line and not line.startswith("#")]
        print(f"1 direct requirement → {len(pins)} pinned packages")
        run([UV, "pip", "compile", "requirements.in", "--format", "pylock.toml", "-o", "pylock.toml",
             "--python-version", "3.12", "--quiet"], cwd=lock_demo)
        print("\nthe same lock in the standard pylock.toml format (first lines, long lines cut):")
        for line in (lock_demo / "pylock.toml").read_text().splitlines()[2:12]:
            print("   " + (line if len(line) < 110 else line[:107] + "..."))

$ uv pip compile requirements.in -o requirements.txt --python-version 3.12 --quiet
# This file was autogenerated by uv via the following command:
#    uv pip compile requirements.in -o requirements.txt --python-version 3.12
certifi==2026.7.22
    # via requests
charset-normalizer==3.5.1
    # via requests
idna==3.19
    # via requests
requests==2.34.2
    # via -r requirements.in
urllib3==2.7.0
    # via requests

1 direct requirement → 5 pinned packages


$ uv pip compile requirements.in --format pylock.toml -o pylock.toml --python-version 3.12 --quiet

the same lock in the standard pylock.toml format (first lines, long lines cut):
   lock-version = "1.0"
   created-by = "uv"
   requires-python = ">=3.12"
   
   [[packages]]
   name = "certifi"
   version = "2026.7.22"
   sdist = { url = "https://files.pythonhosted.org/packages/a3/c2/24167ea9858356b47a87a50d39908bfdb72ceeefe004...
   wheels = [{ url = "https://files.pythonhosted.org/packages/0b/a7/71ac2cff56fec219ed242bb11b8efb69fcc4bec75d...
   


### ✍️ Your Turn

CI tools often need to read a compiled requirements file. Turn `compiled_text` into a dict `{package: version}`, ignoring comment lines and the indented `# via` lines.

In [10]:
compiled_text = """\
# This file was autogenerated by uv via the following command:
#    uv pip compile requirements.in -o requirements.txt
certifi==2026.7.22
    # via requests
idna==3.19
    # via requests
requests==2.34.2
    # via -r requirements.in
"""
pins = None  # TODO: {"certifi": "2026.7.22", ...}
check("pins", pins, {"certifi": "2026.7.22", "idna": "3.19", "requests": "2.34.2"},
      hint="Loop over lines, skip those that start with '#' after stripping, then split on '=='.")

⏳ pins: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
compiled_text = """\
# This file was autogenerated by uv via the following command:
#    uv pip compile requirements.in -o requirements.txt
certifi==2026.7.22
    # via requests
idna==3.19
    # via requests
requests==2.34.2
    # via -r requirements.in
"""
pins = {}
for line in compiled_text.splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    name, version = line.split("==")
    pins[name] = version
check("pins", pins, {"certifi": "2026.7.22", "idna": "3.19", "requests": "2.34.2"})
```
Real files can also contain markers (`; python_version < "3.11"`) and hashes (`--hash=sha256:…`); `packaging.requirements.Requirement` parses those properly.
</details>

> 💡 **Interview angle:** "requirements.txt or lock file?" — libraries declare ranges in `pyproject.toml`; applications commit a lock file with exact transitive versions and hashes, and CI installs *exactly* that. `pip freeze` is a snapshot, not a design.

## 5. uv — The Modern Default 🟡

**uv** (by Astral, written in Rust) replaces a whole toolbox — `pip`, `venv`, `pip-tools`, `pipx`, `pyenv` — with one very fast command that follows the Python packaging standards.

| Task | uv command |
|---|---|
| Start a project (src layout, CLI entry point) | `uv init --package my-app` |
| Add / remove a dependency | `uv add "requests>=2.31"` · `uv add --dev pytest` · `uv remove requests` |
| Resolve and write `uv.lock` | `uv lock` |
| Make `.venv` match the lock exactly | `uv sync` (CI: `uv sync --locked`) |
| Run anything inside the project env | `uv run pytest` · `uv run python train.py` |
| Manage Python versions | `uv python install 3.12` · `uv python list` · `uv python pin 3.12` |
| Run a tool without installing it | `uvx ruff check .` (= `uv tool run ruff check .`) |
| pip-compatible interface | `uv venv` · `uv pip install` · `uv pip compile` |

Let's run the project workflow for real:

In [11]:
uv_project = WORK / "ml-utils"
UV_ENV = clean_env(UV_PYTHON=sys.executable)        # use this notebook's Python version instead of downloading one
uv_project_ok = False

if not UV:
    print("⏭️ Skipped: uv is not installed — see the installation guide linked in ⚙️ Setup.")
else:
    run([UV, "init", "--package", "--name", "ml-utils", "--author-from", "none", "--vcs", "none", uv_project], env=UV_ENV)
    tree(uv_project)
    print("\npyproject.toml:\n" + textwrap.indent((uv_project / "pyproject.toml").read_text(), "   "))

$ uv init --package --name ml-utils --author-from none --vcs none $WORK/ml-utils
  Initialized project `ml-utils` at `$WORK/ml-utils`
📁 ml-utils/
    📄 .python-version
    📄 README.md
    📄 pyproject.toml
    📁 src/
        📁 ml_utils/
            📄 __init__.py

pyproject.toml:
   [project]
   name = "ml-utils"
   version = "0.1.0"
   description = "Add your description here"
   readme = "README.md"
   requires-python = ">=3.12"
   dependencies = []

   [project.scripts]
   ml-utils = "ml_utils:main"

   [build-system]
   requires = ["uv_build>=0.10.3,<0.11.0"]
   build-backend = "uv_build"



In [12]:
if UV:
    added = run([UV, "add", "tomli-w>=1.0"], cwd=uv_project, env=UV_ENV, check=False)
    added_dev = run([UV, "add", "--dev", "pytest>=8"], cwd=uv_project, env=UV_ENV, check=False) if added.returncode == 0 else added
    uv_project_ok = added.returncode == 0 and added_dev.returncode == 0
    if not uv_project_ok:
        print("⏭️ Skipped the rest of the uv demo: `uv add` needs network access to PyPI.")
    else:
        text = (uv_project / "pyproject.toml").read_text()
        print("\npyproject.toml now declares:\n" + textwrap.indent(text[text.index("dependencies"):], "   "))
        lock_text = (uv_project / "uv.lock").read_text()
        print(f"uv.lock: {len(lock_text.splitlines())} lines pinning {lock_text.count('[[package]]')} packages (commit this file!)")

$ uv add tomli-w>=1.0
  Using CPython 3.12.11 interpreter at: ~/Workspaces/Learning-AI-ML/.venv/bin/python
  Creating virtual environment at: .venv
  Resolved 2 packages in 4ms
     Building ml-utils @ file://$WORK/ml-utils
        Built ml-utils @ file://$WORK/ml-utils
  Prepared 1 package in 14ms
  Installed 2 packages in 4ms
   + ml-utils==0.1.0 (from file://$WORK/ml-utils)
   + tomli-w==1.2.0


$ uv add --dev pytest>=8
  Resolved 8 packages in 11ms
     Building ml-utils @ file://$WORK/ml-utils
        Built ml-utils @ file://$WORK/ml-utils
  Prepared 1 package in 6ms
  Uninstalled 1 package in 1ms
  Installed 6 packages in 12ms
   + iniconfig==2.3.0
   ~ ml-utils==0.1.0 (from file://$WORK/ml-utils)
   + packaging==26.3
   + pluggy==1.6.0
   + pygments==2.21.0
   + pytest==9.1.1

pyproject.toml now declares:
   dependencies = [
       "tomli-w>=1.0",
   ]

   [project.scripts]
   ml-utils = "ml_utils:main"

   [build-system]
   requires = ["uv_build>=0.10.3,<0.11.0"]
   build-backend = "uv_build"

   [dependency-groups]
   dev = [
       "pytest>=8",
   ]

uv.lock: 92 lines pinning 8 packages (commit this file!)


In [13]:
if uv_project_ok:
    run([UV, "tree"], cwd=uv_project, env=UV_ENV)
    run([UV, "run", "ml-utils"], cwd=uv_project, env=UV_ENV)          # the console script uv init created
    run([UV, "run", "python", "-c", "import sys, tomli_w; print('running in', sys.prefix); print(tomli_w.dumps({'learning_rate': 0.001}))"],
        cwd=uv_project, env=UV_ENV)

$ uv tree
  ml-utils v0.1.0
  ├── tomli-w v1.2.0
  └── pytest v9.1.1 (group: dev)
      ├── iniconfig v2.3.0
      ├── packaging v26.3
      ├── pluggy v1.6.0
      └── pygments v2.21.0
  Resolved 8 packages in 6ms


$ uv run ml-utils
  Hello from ml-utils!
$ uv run python -c import sys, tomli_w; print('running in', sys.prefix); print(tomli_w.dumps({'learning_rate': 0.001}))
  running in $WORK/ml-utils/.venv
  learning_rate = 0.001


**The CI safety net.** Suppose someone edits `pyproject.toml` by hand but forgets to re-lock. `uv sync --locked` refuses to continue, so CI fails loudly instead of installing something nobody reviewed:

In [14]:
if uv_project_ok:
    pyproject_path = uv_project / "pyproject.toml"
    edited = pyproject_path.read_text().replace('"tomli-w>=1.0",', '"tomli-w>=1.0",\n    "six>=1.16",')
    assert "six>=1.16" in edited, "expected uv add to have written the tomli-w line"
    pyproject_path.write_text(edited)
    run([UV, "sync", "--locked"], cwd=uv_project, env=UV_ENV, check=False)   # ❌ lock is stale → non-zero exit
    run([UV, "lock"], cwd=uv_project, env=UV_ENV)                             # re-resolve and update uv.lock
    run([UV, "sync", "--locked"], cwd=uv_project, env=UV_ENV)                 # ✅ passes now

$ uv sync --locked
  Resolved 9 packages in 8ms
  The lockfile at `uv.lock` needs to be updated, but `--locked` was provided. To update the lockfile, run `uv lock`.
  [exit code 1]
$ uv lock
  Resolved 9 packages in 9ms
  Added six v1.17.0
$ uv sync --locked
  Resolved 9 packages in 6ms
     Building ml-utils @ file://$WORK/ml-utils
        Built ml-utils @ file://$WORK/ml-utils
  Prepared 1 package in 5ms
  Uninstalled 1 package in 1ms
  Installed 2 packages in 3ms
   ~ ml-utils==0.1.0 (from file://$WORK/ml-utils)
   + six==1.17.0


In [15]:
if UV:
    run([UV, "python", "find"], env=clean_env())
    run([UV, "python", "list", "--only-installed"], env=clean_env(), max_lines=5)

$ uv python find
  ~/Workspaces/Learning-AI-ML/.venv/bin/python3


$ uv python list --only-installed
  cpython-3.14.3-macos-aarch64-none     /opt/homebrew/bin/python3.14 -> ../Cellar/python@3.14/3.14.3_1/bin/python3.14
  cpython-3.14.3-macos-aarch64-none     /opt/homebrew/bin/python3 -> ../Cellar/python@3.14/3.14.3_1/bin/python3
  cpython-3.13.1-macos-aarch64-none     /usr/local/bin/python3.13 -> ../../../Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13
  cpython-3.13.1-macos-aarch64-none     /usr/local/bin/python3 -> ../../../Library/Frameworks/Python.framework/Versions/3.13/bin/python3
  cpython-3.13.1-macos-aarch64-none     /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13
  … (7 more lines)


> 💡 **Interview angle:** "`uv sync` vs `uv sync --locked` vs `--frozen`?" — `sync` may update the lock if `pyproject.toml` changed; `--locked` errors if the lock is out of date (use in CI); `--frozen` installs from the lock without checking it against `pyproject.toml`.

## 6. pip-tools, Poetry, and conda 🟢

uv is a great default, but you'll meet other tools at work. An honest comparison:

| Tool | What it is | Choose it when | Watch out |
|---|---|---|---|
| **pip + venv** | built-in installer + environments | minimal setups, anywhere Python exists | no locking by itself |
| **pip-tools** | `pip-compile` → pinned `requirements.txt`, `pip-sync` to apply it | an existing pip workflow that needs locking | one lock per platform/Python |
| **Poetry** | project manager with its own `poetry.lock` | a team already on Poetry | 2.0 (Jan 2025) supports the standard `[project]` table; `poetry shell` was **removed** (use `poetry env activate` or the `poetry-plugin-shell` plugin) |
| **conda** (Miniforge, mamba, pixi) | installs Python *and non-Python binaries* | you need compiled system libraries: CUDA toolkit, GDAL, R, compilers | prefer the community **conda-forge** channel; Anaconda's `defaults` channel falls under Anaconda's Terms of Service, which require a paid plan for for-profit organisations with more than 200 employees |

A conda environment file that avoids the `defaults` channel:

```yaml
# environment.yml  →  conda env create -f environment.yml   (or: mamba / micromamba)
name: geo-ml
channels:
  - conda-forge
  - nodefaults
dependencies:
  - python=3.12
  - gdal            # a C/C++ library that is painful to install with pip
  - pip
  - pip:
      - scikit-learn>=1.5
```

Which of these tools are installed on *this* machine?

In [16]:
for tool in ["uv", "pip", "pip-compile", "poetry", "conda", "mamba", "pixi"]:
    location = shutil.which(tool)
    print(f"{tool:12s} {'✅ ' + short(location) if location else '— not installed'}")

uv           ✅ /opt/homebrew/bin/uv
pip          ✅ /Library/Frameworks/Python.framework/Versions/3.13/bin/pip
pip-compile  — not installed
poetry       — not installed
conda        ✅ /opt/anaconda3/condabin/conda
mamba        — not installed
pixi         — not installed


> 💡 **Interview angle:** "When would you still use conda?" — when the dependency isn't just Python: CUDA toolkits, geospatial or bioinformatics C libraries, R. Mention conda-forge vs Anaconda `defaults` licensing — it shows real-world awareness.

## 7. Modules, Packages, and Namespace Packages 🟡

- A **module** is one `.py` file. `import metrics` runs `metrics.py` once and caches it in `sys.modules`.
- A **package** is a directory of modules you import with dots: `import mlkit.metrics`.
  - A **regular package** has an `__init__.py`, which runs on import and lives in exactly one directory.
  - A **namespace package** (PEP 420, Python 3.3+) has **no** `__init__.py` and can be **split across several directories** — e.g. separately installed plugins `acme.vision` and `acme.nlp`.
- Python searches `sys.path` **in order** and the **first match wins** (so a local `random.py` hides the standard library — ⚠️ Pitfall 4).
- `python -m mlkit` runs `mlkit/__main__.py`; `if __name__ == "__main__":` guards code that should run only when a file is executed directly, not imported.

```
site_a/                         site_b/
├── acme/          (no __init__)└── acme/        (no __init__)   ← one namespace package "acme"
│   └── vision.py                   └── nlp.py
└── mlkit/
    ├── __init__.py   ← regular package
    ├── __main__.py
    └── metrics.py
```

In [17]:
imports_demo = WORK / "imports"
write(imports_demo / "site_a/acme/vision.py", '''
    def detect():
        return "vision plugin"
''')
write(imports_demo / "site_b/acme/nlp.py", '''
    def tokenize(text):
        return text.split()
''')
write(imports_demo / "site_a/mlkit/__init__.py", '''
    print("  (mlkit/__init__.py runs once, on first import)")
    from .metrics import accuracy

    __all__ = ["accuracy"]
''')
write(imports_demo / "site_a/mlkit/metrics.py", '''
    def accuracy(y_true, y_pred):
        return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)
''')
write(imports_demo / "site_a/mlkit/__main__.py", '''
    from mlkit import accuracy

    print("python -m mlkit →", accuracy([1, 0, 1], [1, 1, 1]))
''')

probe = textwrap.dedent('''
    import acme, acme.vision, acme.nlp, mlkit
    print("acme  : namespace package spanning", len(acme.__path__), "directories | __file__ =", acme.__file__)
    print("        plugins from both folders:", acme.vision.detect(), "+", acme.nlp.tokenize("hello world"))
    print("mlkit : regular package | __file__ =", mlkit.__file__.split("imports")[-1])
    print("        accuracy =", mlkit.accuracy([1, 0, 1], [1, 1, 1]))
''').strip()
paths_env = clean_env(PYTHONPATH=os.pathsep.join([str(imports_demo / "site_a"), str(imports_demo / "site_b")]))
run([sys.executable, "-c", probe], env=paths_env)
run([sys.executable, "-m", "mlkit"], env=paths_env);

$ python -c import acme, acme.vision, acme.nlp, mlkit
print("acme  : namespace package spanning", len(acme.__path__), "directories | __file__ =", acme.__file__)
print("        plugins from both folders:", acme.vision.detect(), "+", acme.nlp.tokenize("hello world"))
print("mlkit : regular package | __file__ =", mlkit.__file__.split("imports")[-1])
print("        accuracy =", mlkit.accuracy([1, 0, 1], [1, 1, 1]))
    (mlkit/__init__.py runs once, on first import)
  acme  : namespace package spanning 2 directories | __file__ = None
          plugins from both folders: vision plugin + ['hello', 'world']
  mlkit : regular package | __file__ = /site_a/mlkit/__init__.py
          accuracy = 0.6666666666666666
$ python -m mlkit
    (mlkit/__init__.py runs once, on first import)
  python -m mlkit → 0.6666666666666666


**Distribution name ≠ import name.** You `pip install scikit-learn` but `import sklearn`. `importlib.metadata` can tell you which installed distribution provides an import name:

In [18]:
providers = md.packages_distributions()
for import_name in ["sklearn", "cv2", "yaml", "PIL", "dotenv"]:
    print(f"import {import_name:8s} ← installed by {providers.get(import_name, ['(not installed)'])}")

import sklearn  ← installed by ['scikit-learn']
import cv2      ← installed by ['opencv-python', 'opencv-python-headless']
import yaml     ← installed by ['PyYAML']
import PIL      ← installed by ['pillow']
import dotenv   ← installed by ['python-dotenv']


### ✍️ Your Turn

Create a **namespace package** named `geometry` inside `ns_root`: write `ns_root/geometry/area.py` containing `def square(side): return side * side` — and **no** `__init__.py`. The check imports it in a fresh Python process.

In [19]:
ns_root = WORK / "your_turn_ns"
# TODO: create the file ns_root / "geometry" / "area.py" (hint: Path.mkdir(parents=True) and Path.write_text)

got = None
if (ns_root / "geometry" / "area.py").exists():
    probe_result = run([sys.executable, "-c", "import geometry.area as a, geometry; print(a.square(3), geometry.__file__, "
                        "(__import__('pathlib').Path(geometry.__path__[0]) / '__init__.py').exists())"], cwd=ns_root, show=False, check=False)
    got = probe_result.stdout.split()
check("namespace_package", got, ["9", "None", "False"], hint="Only area.py should exist inside geometry/ — no __init__.py.")

⏳ namespace_package: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
ns_root = WORK / "your_turn_ns"
(ns_root / "geometry").mkdir(parents=True, exist_ok=True)
(ns_root / "geometry" / "area.py").write_text("def square(side):\n    return side * side\n")

probe_result = run([sys.executable, "-c", "import geometry.area as a, geometry; print(a.square(3), geometry.__file__, "
                    "(__import__('pathlib').Path(geometry.__path__[0]) / '__init__.py').exists())"], cwd=ns_root, show=False, check=False)
check("namespace_package", probe_result.stdout.split(), ["9", "None", "False"])
```
`geometry.__file__` is `None` because a namespace package has no single file behind it.
</details>

> 💡 **Interview angle:** "Does a directory need `__init__.py` to be importable?" — no, not since Python 3.3 (PEP 420 namespace packages). You still usually want `__init__.py` for your own packages: it's explicit, runs initialisation, and tools like `setuptools.find_packages` look for it.

## 8. Project Layout and pyproject.toml 🟡

**The `src/` layout** puts your package one level down:

```
hello-ml/
├── pyproject.toml      ← metadata, dependencies, build backend, tool config
├── README.md
├── src/
│   └── hello_ml/       ← the import package (underscore); distribution name is hello-ml
│       ├── __init__.py
│       └── cli.py
└── tests/
    └── test_greet.py
```

Because `src/` isn't on `sys.path`, you **can't accidentally import the un-installed folder** — tests must use the *installed* package, so packaging mistakes show up on your machine instead of your users'.

**`pyproject.toml` tables:**

| Table | Standard | Purpose |
|---|---|---|
| `[build-system]` | PEP 517/518 | which **build backend** turns the source into a wheel, and what it needs |
| `[project]` | PEP 621 | name, version, `requires-python`, `dependencies`, license, readme |
| `[project.optional-dependencies]` | PEP 621 | **extras**: `pip install "hello-ml[plot]"` |
| `[project.scripts]` | PEP 621 | **console commands** created on install |
| `[dependency-groups]` | PEP 735 | dev-only groups (tests, linters) — never published |
| `[tool.*]` | per tool | settings for pytest, ruff, uv, setuptools, … |

Correct `build-backend` values: `setuptools.build_meta` (setuptools) · `hatchling.build` (Hatch) · `uv_build` (uv) · `flit_core.buildapi` (Flit) · `poetry.core.masonry.api` (Poetry).

In [20]:
hello = WORK / "hello-ml"
write(hello / "pyproject.toml", '''
    [build-system]
    requires = ["setuptools>=77"]
    build-backend = "setuptools.build_meta"

    [project]
    name = "hello-ml"
    version = "0.1.0"
    description = "A tiny package to learn packaging"
    readme = "README.md"
    requires-python = ">=3.10"
    license = "MIT"
    dependencies = []

    [project.optional-dependencies]
    plot = ["matplotlib>=3.8"]

    [project.scripts]
    hello-ml = "hello_ml.cli:main"

    [dependency-groups]
    dev = ["pytest>=8"]

    [tool.pytest.ini_options]
    testpaths = ["tests"]
''')
write(hello / "README.md", "# hello-ml\n\nA tiny package used to learn packaging.\n")
write(hello / "src/hello_ml/__init__.py", '''
    """hello_ml — the smallest useful package."""
    from importlib.metadata import version

    __version__ = version("hello-ml")


    def greet(name: str) -> str:
        return f"Hello, {name}! Welcome to packaging."
''')
write(hello / "src/hello_ml/cli.py", '''
    import argparse

    from hello_ml import __version__, greet


    def main(argv=None):
        parser = argparse.ArgumentParser(prog="hello-ml", description="Say hello.")
        parser.add_argument("name", nargs="?", default="world")
        parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
        args = parser.parse_args(argv)
        print(greet(args.name))
        return 0
''')
write(hello / "tests/test_greet.py", '''
    import pytest

    from hello_ml import greet
    from hello_ml.cli import main


    def test_greet_includes_name():
        assert greet("Ada") == "Hello, Ada! Welcome to packaging."


    @pytest.mark.parametrize("name", ["Ada", "Grace", "Linus"])
    def test_cli_prints_greeting(name, capsys):
        assert main([name]) == 0
        assert name in capsys.readouterr().out
''')
tree(hello)

config = tomllib.loads((hello / "pyproject.toml").read_text())
print("\nbuild backend :", config["build-system"]["build-backend"])
print("name, version :", config["project"]["name"], config["project"]["version"])
print("console script:", config["project"]["scripts"])
print("extras        :", config["project"]["optional-dependencies"])

print("\nsrc layout protection — importing from the project root BEFORE installing fails:")
run([sys.executable, "-c", "import hello_ml"], cwd=hello, check=False, tail=True, max_lines=1);

📁 hello-ml/
    📄 README.md
    📄 pyproject.toml
    📁 src/
        📁 hello_ml/
            📄 __init__.py
            📄 cli.py
    📁 tests/
        📄 test_greet.py

build backend : setuptools.build_meta
name, version : hello-ml 0.1.0
console script: {'hello-ml': 'hello_ml.cli:main'}
extras        : {'plot': ['matplotlib>=3.8']}

src layout protection — importing from the project root BEFORE installing fails:
$ python -c import hello_ml
  … (2 earlier lines hidden)
  ModuleNotFoundError: No module named 'hello_ml'
  [exit code 1]


### ✍️ Your Turn

The old version of this notebook used `build-backend = "setuptools.backends.legacy:build"`, which doesn't exist. Write `build_system_toml`: a TOML string with a `[build-system]` table that requires `setuptools>=77` and names a backend module that can actually be imported.

In [21]:
build_system_toml = None  # TODO: a multi-line TOML string


def module_exists(dotted_name):
    try:
        return importlib.util.find_spec(dotted_name) is not None
    except ModuleNotFoundError:          # a missing parent package raises instead of returning None
        return False


got = None
if build_system_toml is not None:
    table = tomllib.loads(build_system_toml)["build-system"]
    got = [any(r.startswith("setuptools") for r in table["requires"]), module_exists(table["build-backend"].split(":")[0])]
check("build_system", got, [True, True], hint='requires = ["setuptools>=77"] and build-backend = "setuptools.build_meta"')

⏳ build_system: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
build_system_toml = """
[build-system]
requires = ["setuptools>=77"]
build-backend = "setuptools.build_meta"
"""


def module_exists(dotted_name):
    try:
        return importlib.util.find_spec(dotted_name) is not None
    except ModuleNotFoundError:
        return False


table = tomllib.loads(build_system_toml)["build-system"]
got = [any(r.startswith("setuptools") for r in table["requires"]), module_exists(table["build-backend"].split(":")[0])]
print("the old backend exists?", module_exists("setuptools.backends.legacy"))
check("build_system", got, [True, True])
```
</details>

> 💡 **Interview angle:** "What goes in `pyproject.toml`?" — `[build-system]` (how to build: PEP 517), `[project]` (standard metadata: PEP 621), dependency groups (PEP 735), and `[tool.*]` settings. `setup.py` is no longer needed for pure-Python packages.

## 9. Building and Installing — Wheels, sdists, Editable Installs 🟡

- **Building** = running the backend to produce distribution files. `python -m build` (or `uv build`) makes both:
  - an **sdist** `hello_ml-0.1.0.tar.gz` — source + metadata; installing it requires a build step.
  - a **wheel** `hello_ml-0.1.0-py3-none-any.whl` — a zip that installers simply unpack. The name encodes **tags**: `{name}-{version}-{python}-{abi}-{platform}.whl`. `py3-none-any` = pure Python; `cp312-cp312-manylinux_2_28_x86_64` = compiled for CPython 3.12 on Linux x86-64.
- **Installing** a wheel = unpacking into site-packages + generating **console-script launchers** from `[project.scripts]`.
- An **editable install** (`pip install -e .`, PEP 660) points the environment at your source folder, so edits take effect without reinstalling.

In [22]:
start = time.perf_counter()
built = run([sys.executable, "-m", "build", "--outdir", hello / "dist", hello], check=False, tail=True, max_lines=2)
if built.returncode != 0:
    print("⚠️ The isolated build needs PyPI access to fetch setuptools. Retrying with --no-isolation "
          "(uses the setuptools already installed here).")
    run([sys.executable, "-m", "build", "--no-isolation", "--outdir", hello / "dist", hello], tail=True, max_lines=2)
wheel = next((hello / "dist").glob("*.whl"))
sdist = next((hello / "dist").glob("*.tar.gz"))
print(f"\nbuilt in {time.perf_counter() - start:.1f} s: {wheel.name} + {sdist.name}")

name, version, python_tag, abi_tag, platform_tag = wheel.stem.split("-")
print(f"wheel tags → python={python_tag} abi={abi_tag} platform={platform_tag}  (pure Python: installs anywhere)")

with zipfile.ZipFile(wheel) as zf:
    print("\nwheel contents:")
    for member in zf.namelist():
        print("   ", member)
    dist_info = f"{name}-{version}.dist-info"
    print("\nMETADATA (first lines):\n" + textwrap.indent("\n".join(zf.read(f"{dist_info}/METADATA").decode().splitlines()[:7]), "   "))
    print("entry_points.txt:", zf.read(f"{dist_info}/entry_points.txt").decode().strip().replace("\n", "  |  "))

with tarfile.open(sdist) as tf:
    print("\nsdist files:", sorted(m.name.split("/", 1)[1] for m in tf.getmembers() if m.isfile()))

$ python -m build --outdir $WORK/hello-ml/dist $WORK/hello-ml
  … (96 earlier lines hidden)
    - setuptools==84.0.0
  * Building wheel...

built in 6.1 s: hello_ml-0.1.0-py3-none-any.whl + hello_ml-0.1.0.tar.gz
wheel tags → python=py3 abi=none platform=any  (pure Python: installs anywhere)

wheel contents:
    hello_ml/__init__.py
    hello_ml/cli.py
    hello_ml-0.1.0.dist-info/METADATA
    hello_ml-0.1.0.dist-info/WHEEL
    hello_ml-0.1.0.dist-info/entry_points.txt
    hello_ml-0.1.0.dist-info/top_level.txt
    hello_ml-0.1.0.dist-info/RECORD

METADATA (first lines):
   Metadata-Version: 2.4
   Name: hello-ml
   Version: 0.1.0
   Summary: A tiny package to learn packaging
   License-Expression: MIT
   Requires-Python: >=3.10
   Description-Content-Type: text/markdown
entry_points.txt: [console_scripts]  |  hello-ml = hello_ml.cli:main

sdist files: ['PKG-INFO', 'README.md', 'pyproject.toml', 'setup.cfg', 'src/hello_ml.egg-info/PKG-INFO', 'src/hello_ml.egg-info/SOURCES.txt', 'src/hel

In [23]:
wheel_venv = WORK / "venv-wheel"
wheel_python = make_venv(wheel_venv)
pip_install(wheel_python, wheel)

hello_cmd = venv_bin(wheel_venv, "hello-ml")
run([hello_cmd, "Ada"])
run([hello_cmd, "--version"])
if hello_cmd.suffix != ".exe":
    print("the generated launcher is a tiny Python script:\n" + textwrap.indent(short(hello_cmd.read_text()), "   "))
run([wheel_python, "-c", "from importlib.metadata import version, entry_points; "
     "print('installed version:', version('hello-ml')); print(entry_points(group='console_scripts', name='hello-ml'))"])
print("\nthe notebook's own environment is untouched:")
run([sys.executable, "-c", "import hello_ml"], check=False, tail=True, max_lines=1);

$ uv pip install --python $WORK/venv-wheel/bin/python $WORK/hello-ml/dist/hello_ml-0.1.0-py3-none-any.whl
  Using Python 3.12.11 environment at: $WORK/venv-wheel
  Resolved 1 package in 2ms
  Prepared 1 package in 3ms
  Installed 1 package in 3ms
   + hello-ml==0.1.0 (from file://$WORK/hello-ml/dist/hello_ml-0.1.0-py3-none-any.whl)


$ $WORK/venv-wheel/bin/hello-ml Ada
  Hello, Ada! Welcome to packaging.
$ $WORK/venv-wheel/bin/hello-ml --version
  hello-ml 0.1.0
the generated launcher is a tiny Python script:
   #!$WORK/venv-wheel/bin/python
   # -*- coding: utf-8 -*-
   import sys
   from hello_ml.cli import main
   if __name__ == "__main__":
       if sys.argv[0].endswith("-script.pyw"):
           sys.argv[0] = sys.argv[0][:-11]
       elif sys.argv[0].endswith(".exe"):
           sys.argv[0] = sys.argv[0][:-4]
       sys.exit(main())



$ $WORK/venv-wheel/bin/python -c from importlib.metadata import version, entry_points; print('installed version:', version('hello-ml')); print(entry_points(group='console_scripts', name='hello-ml'))
  installed version: 0.1.0
  (EntryPoint(name='hello-ml', value='hello_ml.cli:main', group='console_scripts'),)

the notebook's own environment is untouched:
$ python -c import hello_ml
  … (2 earlier lines hidden)
  ModuleNotFoundError: No module named 'hello_ml'
  [exit code 1]


**Editable install:** change the source and the change is live immediately. (The wheel install above is a *copy*, so it keeps the old text.)

In [24]:
editable_venv = WORK / "venv-editable"
editable_python = make_venv(editable_venv)
pip_install(editable_python, "-e", hello)
editable_cmd = venv_bin(editable_venv, "hello-ml")
run([editable_cmd, "Grace"])

init_file = hello / "src/hello_ml/__init__.py"
original_source = init_file.read_text()
init_file.write_text(original_source.replace("Welcome to packaging.", "Your edit is live — no reinstall!"))
run([editable_cmd, "Grace"])                 # editable → sees the edit
run([hello_cmd, "Grace"])                    # regular wheel install → still the old copy
init_file.write_text(original_source)        # restore for the next sections

site_packages = next(editable_venv.rglob("site-packages"))
print("what the editable install placed in site-packages:", sorted(p.name for p in site_packages.iterdir() if "hello" in p.name.lower()))

$ uv pip install --python $WORK/venv-editable/bin/python -e $WORK/hello-ml
  Using Python 3.12.11 environment at: $WORK/venv-editable
  Resolved 1 package in 1ms
     Building hello-ml @ file://$WORK/hello-ml
        Built hello-ml @ file://$WORK/hello-ml
  Prepared 1 package in 835ms
  Installed 1 package in 3ms
   + hello-ml==0.1.0 (from file://$WORK/hello-ml)


$ $WORK/venv-editable/bin/hello-ml Grace
  Hello, Grace! Welcome to packaging.
$ $WORK/venv-editable/bin/hello-ml Grace
  Hello, Grace! Your edit is live — no reinstall!
$ $WORK/venv-wheel/bin/hello-ml Grace
  Hello, Grace! Welcome to packaging.


what the editable install placed in site-packages: ['__editable__.hello_ml-0.1.0.pth', 'hello_ml-0.1.0.dist-info']


### ✍️ Your Turn

Parse a wheel filename into its parts. Return a dict with keys `name`, `version`, `python`, `abi`, `platform` for `wheel_file` (assume there's no optional build tag).

In [25]:
wheel_file = "numpy-2.1.3-cp312-cp312-macosx_14_0_arm64.whl"
wheel_parts = None  # TODO
check("wheel_parts", wheel_parts,
      {"name": "numpy", "version": "2.1.3", "python": "cp312", "abi": "cp312", "platform": "macosx_14_0_arm64"},
      hint="Remove the .whl suffix, then split on '-' into exactly five pieces.")

⏳ wheel_parts: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
wheel_file = "numpy-2.1.3-cp312-cp312-macosx_14_0_arm64.whl"
keys = ["name", "version", "python", "abi", "platform"]
wheel_parts = dict(zip(keys, wheel_file.removesuffix(".whl").split("-")))
check("wheel_parts", wheel_parts,
      {"name": "numpy", "version": "2.1.3", "python": "cp312", "abi": "cp312", "platform": "macosx_14_0_arm64"})
```
Names can't contain `-` inside a wheel filename (they're normalised to `_`), which is why a simple split works. `packaging.utils.parse_wheel_filename` handles build tags too.
</details>

> 💡 **Interview angle:** "Wheel vs sdist?" — a wheel is pre-built and just unpacked (fast, no compiler, platform-tagged); an sdist must be built on the target machine. Publish both; installers prefer a compatible wheel.

## 10. Testing with pytest 🟢

**pytest** finds files named `test_*.py` and functions named `test_*`, and uses plain `assert` with detailed failure messages.

- **Fixtures** are ready-made helpers passed in by argument name: `tmp_path` (a temp folder), `capsys` (captured output), `monkeypatch` (patch env vars).
- `@pytest.mark.parametrize` runs one test with many inputs; `pytest.raises` asserts an exception.
- With a `src/` layout, run tests **against the installed package** — that's what we do here, in the wheel venv.

In [26]:
pytest_ready = pip_install(wheel_python, "pytest>=8", check=False).returncode == 0
if not pytest_ready:
    print("⏭️ Skipped: couldn't install pytest into the test venv (no network?).")
else:
    run([wheel_python, "-m", "pytest", hello / "tests"], cwd=WORK)

$ uv pip install --python $WORK/venv-wheel/bin/python pytest>=8
  Using Python 3.12.11 environment at: $WORK/venv-wheel
  Resolved 5 packages in 2ms
  Installed 5 packages in 11ms
   + iniconfig==2.3.0
   + packaging==26.3
   + pluggy==1.6.0
   + pygments==2.21.0
   + pytest==9.1.1


$ $WORK/venv-wheel/bin/python -m pytest $WORK/hello-ml/tests
  ============================= test session starts ==============================
  platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0
  rootdir: $WORK/hello-ml
  configfile: pyproject.toml
  collected 4 items
  
  hello-ml/tests/test_greet.py ....                                        [100%]
  
  ============================== 4 passed in 0.01s ===============================


And what a failure looks like — pytest shows both sides of the comparison:

In [27]:
if pytest_ready:
    failing_test = write(hello / "tests/test_failing_demo.py", '''
        from hello_ml import greet


        def test_shouting():
            assert greet("Ada") == "HELLO, ADA!"
    ''')
    run([wheel_python, "-m", "pytest", "-q", "--tb=short", failing_test], cwd=WORK, check=False, max_lines=14)
    failing_test.unlink()

$ $WORK/venv-wheel/bin/python -m pytest -q --tb=short $WORK/hello-ml/tests/test_failing_demo.py
  F                                                                        [100%]
  =================================== FAILURES ===================================
  ________________________________ test_shouting _________________________________
  hello-ml/tests/test_failing_demo.py:5: in test_shouting
      assert greet("Ada") == "HELLO, ADA!"
  E   AssertionError: assert 'Hello, Ada! ...to packaging.' == 'HELLO, ADA!'
  E     
  E     - HELLO, ADA!
  E     + Hello, Ada! Welcome to packaging.
  =========================== short test summary info ============================
  FAILED hello-ml/tests/test_failing_demo.py::test_shouting - AssertionError: a...
  1 failed in 0.02s
  [exit code 1]


> 💡 **Interview angle:** "How do you test an ML package?" — unit-test pure functions (feature transforms, metrics) with tiny hand-checkable inputs, use `tmp_path` for file I/O, parametrize edge cases (empty input, NaNs), and run tests against the installed package in CI.

## 11. Linting and Formatting with Ruff 🟢

**Ruff** is a very fast linter (finds bugs and style problems) and formatter (rewrites layout), replacing flake8, isort, and most of Black. Configure it in `pyproject.toml` — rule selection belongs under **`[tool.ruff.lint]`** (the old top-level `select` is deprecated):

```toml
[tool.ruff]
line-length = 100
target-version = "py312"

[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP", "SIM"]   # pycodestyle, pyflakes, isort, bugbear, pyupgrade, simplify
```

In [28]:
lint_demo = WORK / "lint-demo"
write(lint_demo / "train.py", '''
    import os
    import sys


    def load(path, cache={}):
        l = open(path).read()
        return l if l != None else ""


    def scale(x):
        return [v*2 for v in x ]
''')
write(lint_demo / "pyproject.toml", '''
    [tool.ruff]
    line-length = 100
    target-version = "py312"

    [tool.ruff.lint]
    select = ["E", "F", "I", "B", "UP", "SIM"]
''')

if not UV:
    print("⏭️ Skipped: needs uv to run ruff without installing it (`uvx ruff`). Alternatively: `pip install ruff`.")
else:
    ruff_check = run([UV, "tool", "run", "ruff", "check", "--output-format", "concise", "."], cwd=lint_demo, check=False)
    if "error" in ruff_check.stderr.lower() and "train.py" not in ruff_check.stdout:
        print("⏭️ Skipped: uv couldn't download ruff (offline?).")
    else:
        run([UV, "tool", "run", "ruff", "format", "--diff", "."], cwd=lint_demo, check=False, max_lines=12)
        run([UV, "tool", "run", "ruff", "--version"], cwd=lint_demo)

$ uv tool run ruff check --output-format concise .
  train.py:1:8: F401 [*] `os` imported but unused
  train.py:2:8: F401 [*] `sys` imported but unused
  train.py:5:22: B006 Do not use mutable data structures for argument defaults
  train.py:6:5: E741 Ambiguous variable name: `l`
  train.py:6:9: SIM115 Use a context manager for opening files
  train.py:7:22: E711 Comparison to `None` should be `cond is not None`
  Found 6 errors.
  [*] 2 fixable with the `--fix` option (2 hidden fixes can be enabled with the `--unsafe-fixes` option).
  [exit code 1]


$ uv tool run ruff format --diff .
  --- train.py
  +++ train.py
  @@ -8,4 +8,4 @@
   
   
   def scale(x):
  -    return [v*2 for v in x ]
  +    return [v * 2 for v in x]
  
  1 file would be reformatted
  [exit code 1]
$ uv tool run ruff --version
  ruff 0.16.7


The deprecated config style still runs, but warns:

In [29]:
old_style = WORK / "lint-old-config"
write(old_style / "pyproject.toml", '''
    [tool.ruff]
    select = ["E", "F"]
''')
shutil.copy(lint_demo / "train.py", old_style / "train.py")
if UV:
    old_run = run([UV, "tool", "run", "ruff", "check", "--output-format", "concise", "."], cwd=old_style, check=False, show=False)
    warning_lines = [re.sub(r"\x1b\[[0-9;]*m", "", line) for line in old_run.stderr.splitlines() if line.strip()]
    print("$ ruff check .   ← with `select` at the top level of [tool.ruff]")
    print("\n".join("  " + line for line in warning_lines[:4]) or "  (no warning printed by this ruff version)")

$ ruff check .   ← with `select` at the top level of [tool.ruff]
    - 'select' -> 'lint.select'


> 💡 **Interview angle:** "What does your CI run on every PR?" — `uv sync --locked`, `ruff check`, `ruff format --check`, and `pytest` (often plus a type checker). Lint rules like bugbear's `B006` catch real bugs such as mutable default arguments.

## 12. Reproducible ML Environments 🔴

"Same code, same data, different result" usually comes from the environment. Reproducibility has layers:

| Layer | Pin it with |
|---|---|
| Python version | `requires-python`, `.python-version` (`uv python pin 3.12`) |
| Direct dependencies | `[project] dependencies` with sensible ranges |
| *All* dependencies | a committed lock file (`uv.lock` / `pylock.toml`) + `uv sync --locked` |
| Platform-specific wheels | environment markers and per-platform indexes (CUDA vs CPU PyTorch) |
| System libraries & drivers | a Docker base image (e.g. a CUDA runtime image), documented driver versions |
| Randomness | seeds and deterministic settings (covered in the deep-learning notebooks) |

**PyTorch is the classic trap.** The same `torch` version comes in different builds: according to uv's PyTorch guide, PyPI hosts CPU-only wheels for Windows and macOS and CUDA-enabled wheels for Linux (CUDA 13.0 as of PyTorch 2.11), while PyTorch's own indexes (`https://download.pytorch.org/whl/cpu`, `…/whl/cu130`, …) host each variant. With uv you choose per platform:

```toml
[tool.uv.sources]
torch = [
  { index = "pytorch-cpu", marker = "sys_platform != 'linux'" },
  { index = "pytorch-cu130", marker = "sys_platform == 'linux'" },
]

[[tool.uv.index]]
name = "pytorch-cpu"
url = "https://download.pytorch.org/whl/cpu"
explicit = true          # only used for packages that ask for it

[[tool.uv.index]]
name = "pytorch-cu130"
url = "https://download.pytorch.org/whl/cu130"
explicit = true
```

(`uv pip install torch --torch-backend=auto` picks a matching build automatically in the pip interface.) Other ML libraries need system pieces too: e.g. LightGBM and XGBoost on macOS need OpenMP (`brew install libomp`).

A useful habit: **save an environment fingerprint with every trained model.**

In [30]:
from packaging import tags


def environment_fingerprint(packages=("numpy", "scikit-learn", "torch")):
    info = {
        "python": platform.python_version(),
        "implementation": platform.python_implementation(),
        "os": platform.system(),
        "machine": platform.machine(),
        "packages": {p: dist_version(p) for p in packages},
        "best_wheel_tag": str(next(iter(tags.sys_tags()))),
    }
    try:
        import torch
        info["torch_cuda_build"] = torch.version.cuda           # None → a CPU (or MPS) build
        info["cuda_available"] = torch.cuda.is_available()
        info["mps_available"] = torch.backends.mps.is_available()
    except ImportError:
        info["torch"] = "not installed"
    if uv_project_ok:
        info["uv_lock_sha256"] = hashlib.sha256((uv_project / "uv.lock").read_bytes()).hexdigest()[:16]
    return info


fingerprint = environment_fingerprint()
print(json.dumps(fingerprint, indent=2))
(OUTPUT_DIR / "environment_fingerprint.json").write_text(json.dumps(fingerprint, indent=2))
print("saved →", OUTPUT_DIR / "environment_fingerprint.json")

{
  "python": "3.12.11",
  "implementation": "CPython",
  "os": "Darwin",
  "machine": "arm64",
  "packages": {
    "numpy": "2.5.3",
    "scikit-learn": "1.9.1",
    "torch": "2.14.0"
  },
  "best_wheel_tag": "cp312-cp312-macosx_26_0_arm64",
  "torch_cuda_build": null,
  "cuda_available": false,
  "mps_available": true,
  "uv_lock_sha256": "cb1732777c3202e2"
}
saved → _outputs/environment_fingerprint.json


Lock resolvers decide per platform by **evaluating markers** against each target environment — exactly what `packaging` does:

In [31]:
cuda_marker = Marker("sys_platform == 'linux'")
for target in ["linux", "darwin", "win32"]:
    print(f"{target:7s} → torch from {'pytorch-cu130' if cuda_marker.evaluate({'sys_platform': target}) else 'pytorch-cpu'}")

supported = list(tags.sys_tags())
print(f"\nthis interpreter accepts {len(supported)} wheel tags; most preferred: {[str(t) for t in supported[:3]]}")

linux   → torch from pytorch-cu130
darwin  → torch from pytorch-cpu
win32   → torch from pytorch-cpu

this interpreter accepts 1230 wheel tags; most preferred: ['cp312-cp312-macosx_26_0_arm64', 'cp312-cp312-macosx_26_0_universal2', 'cp312-cp312-macosx_25_0_arm64']


> 💡 **Interview angle:** "`torch.cuda.is_available()` is `False` on a GPU box — what do you check?" — `torch.version.cuda` (is it a CPU build?), which index it came from, and whether the NVIDIA driver (`nvidia-smi`) supports that CUDA version. Then fix it in the lock file, not by hand.

## 13. .gitignore and Secrets 🟢

**Commit:** `pyproject.toml`, the lock file, `.python-version`, `.env.example`, source, tests.
**Never commit:** `.venv/`, `__pycache__/`, `dist/`, `build/`, `*.egg-info/`, caches, large data and model files, and above all **secrets** (`.env`).

The **twelve-factor** rule: configuration and secrets come from **environment variables**. Locally, keep them in a git-ignored `.env` file loaded by `python-dotenv` (or pydantic-settings); in production use your platform's secret manager. If a key ever lands in git, **rotate it** — deleting the commit doesn't remove it from history or from forks.

In [32]:
secrets_demo = WORK / "secrets-demo"
write(secrets_demo / ".env.example", '''
    # Copy to .env and fill in real values. This template is committed; .env is not.
    OPENAI_API_KEY=
    MLFLOW_TRACKING_URI=http://localhost:5000
''')
write(secrets_demo / ".env", '''
    OPENAI_API_KEY="demo-not-a-real-key-0123456789abcdef"
    MLFLOW_TRACKING_URI=http://localhost:5000
''')
write(secrets_demo / ".gitignore", '''
    # environments & caches
    .venv/
    __pycache__/
    .pytest_cache/
    .ruff_cache/
    # build output
    build/
    dist/
    *.egg-info/
    # secrets & local config
    .env
    # data, models, run outputs
    data/
    *.ckpt
    *.pt
    mlruns/
''')


def mask(secret):
    """Never print secrets — show just enough to recognise which key is loaded."""
    return f"{secret[:4]}…{secret[-4:]} ({len(secret)} chars)" if secret and len(secret) > 8 else "***"


try:
    from dotenv import dotenv_values
except ImportError:
    print("⏭️ Skipped: `pip install python-dotenv` to load .env files.")
else:
    values = dotenv_values(secrets_demo / ".env")
    print("loaded from .env:", {k: (mask(v) if "KEY" in k else v) for k, v in values.items()})

for var in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "HF_TOKEN"]:
    print(f"{var:18s} is {'set' if os.environ.get(var) else 'not set'} in this process")
print("\n.gitignore protects .env:", ".env" in (secrets_demo / ".gitignore").read_text().split())

loaded from .env: {'OPENAI_API_KEY': 'demo…cdef (36 chars)', 'MLFLOW_TRACKING_URI': 'http://localhost:5000'}
OPENAI_API_KEY     is not set in this process
ANTHROPIC_API_KEY  is not set in this process
HF_TOKEN           is not set in this process

.gitignore protects .env: True


> 💡 **Interview angle:** "How do you manage API keys?" — environment variables; `.env` locally (git-ignored, with a committed `.env.example`); a secret manager in production; mask secrets in logs; rotate on leak; add a secret scanner (e.g. gitleaks) to pre-commit or CI.

## 🔧 Build It From Scratch

### A wheel by hand

A wheel is "just a zip file" — let's prove it. We'll build `hello-ml` **without setuptools**, using only `zipfile`, `hashlib`, and `base64`, install it with the real installer, run its command, and compare it with the setuptools-built wheel.

A wheel contains the package files plus a `{name}-{version}.dist-info/` folder with:
- `METADATA` — name, version (the "label on the box")
- `WHEEL` — wheel format version and tags
- `entry_points.txt` — console scripts
- `RECORD` — every file with its **sha256 hash** (URL-safe base64, no `=` padding) and size, so installers can verify and uninstall

In [33]:
def record_hash(data: bytes) -> str:
    digest = hashlib.sha256(data).digest()
    return "sha256=" + base64.urlsafe_b64encode(digest).rstrip(b"=").decode()


def build_wheel_by_hand(package_dir, dist_name, version, console_scripts, out_dir):
    normalized = dist_name.replace("-", "_")
    dist_info = f"{normalized}-{version}.dist-info"
    files = {}
    for path in sorted(Path(package_dir).rglob("*.py")):
        files[f"{Path(package_dir).name}/{path.relative_to(package_dir).as_posix()}"] = path.read_bytes()
    files[f"{dist_info}/METADATA"] = f"Metadata-Version: 2.1\nName: {dist_name}\nVersion: {version}\n".encode()
    files[f"{dist_info}/WHEEL"] = b"Wheel-Version: 1.0\nGenerator: by-hand (notebook)\nRoot-Is-Purelib: true\nTag: py3-none-any\n"
    files[f"{dist_info}/entry_points.txt"] = ("[console_scripts]\n" + "".join(f"{k} = {v}\n" for k, v in console_scripts.items())).encode()
    record = [f"{arcname},{record_hash(data)},{len(data)}" for arcname, data in files.items()]
    record.append(f"{dist_info}/RECORD,,")                     # RECORD can't contain its own hash
    files[f"{dist_info}/RECORD"] = ("\n".join(record) + "\n").encode()

    wheel_path = Path(out_dir) / f"{normalized}-{version}-py3-none-any.whl"
    wheel_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(wheel_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for arcname, data in files.items():
            zf.writestr(arcname, data)
    return wheel_path


hand_wheel = build_wheel_by_hand(hello / "src/hello_ml", "hello-ml", "0.1.0", {"hello-ml": "hello_ml.cli:main"}, WORK / "hand-dist")
print("built:", hand_wheel.name, f"({hand_wheel.stat().st_size} bytes)")

with zipfile.ZipFile(hand_wheel) as ours, zipfile.ZipFile(wheel) as theirs:
    def package_records(zf):
        record_name = next(n for n in zf.namelist() if n.endswith(".dist-info/RECORD"))
        rows = [line.split(",") for line in zf.read(record_name).decode().splitlines() if line]
        return {path: digest for path, digest, _ in rows if not path.startswith("hello_ml-0.1.0.dist-info")}

    ours_records, their_records = package_records(ours), package_records(theirs)
    print("package files (ours)      :", sorted(ours_records))
    print("package files (setuptools):", sorted(their_records))
    assert ours_records == their_records
    for path, digest in ours_records.items():
        assert digest == record_hash(ours.read(path))
print("✅ same files with identical sha256 hashes as the setuptools wheel, and every RECORD hash verifies")

built: hello_ml-0.1.0-py3-none-any.whl (1667 bytes)
package files (ours)      : ['hello_ml/__init__.py', 'hello_ml/cli.py']
package files (setuptools): ['hello_ml/__init__.py', 'hello_ml/cli.py']
✅ same files with identical sha256 hashes as the setuptools wheel, and every RECORD hash verifies


In [34]:
hand_venv = WORK / "venv-hand-wheel"
hand_python = make_venv(hand_venv)
pip_install(hand_python, hand_wheel)
by_hand = run([venv_bin(hand_venv, "hello-ml"), "Ada"]).stdout
by_setuptools = run([hello_cmd, "Ada"], show=False).stdout
assert by_hand == by_setuptools
run([hand_python, "-c", "from importlib.metadata import version; print('metadata read back:', version('hello-ml'))"])
print("✅ our hand-made wheel installs and its console script behaves exactly like the setuptools build")

$ uv pip install --python $WORK/venv-hand-wheel/bin/python $WORK/hand-dist/hello_ml-0.1.0-py3-none-any.whl
  Using Python 3.12.11 environment at: $WORK/venv-hand-wheel
  Resolved 1 package in 1ms
  Prepared 1 package in 2ms
  Installed 1 package in 3ms
   + hello-ml==0.1.0 (from file://$WORK/hand-dist/hello_ml-0.1.0-py3-none-any.whl)


$ $WORK/venv-hand-wheel/bin/hello-ml Ada
  Hello, Ada! Welcome to packaging.


$ $WORK/venv-hand-wheel/bin/python -c from importlib.metadata import version; print('metadata read back:', version('hello-ml'))
  metadata read back: 0.1.0
✅ our hand-made wheel installs and its console script behaves exactly like the setuptools build


## ⚠️ Common Pitfalls

### ❌ Pitfall 1 — Unquoted version specifiers in a shell

We use `echo` so nothing is really installed; the shell treats `>` the same way either way.

In [35]:
quote_demo = WORK / "quoting"
quote_demo.mkdir()
subprocess.run("echo pip install numpy>=1.24", shell=True, cwd=quote_demo, check=True)    # ❌
created = sorted(p.name for p in quote_demo.iterdir())
print("❌ files created by the unquoted command:", created, "| containing:", (quote_demo / "=1.24").read_text().strip())
print("   → pip would install the LATEST numpy with no constraint, and hide its output in that file")

quoted = subprocess.run('echo pip install "numpy>=1.24"', shell=True, cwd=quote_demo, capture_output=True, text=True, check=True)
print("✅ quoted, the command receives:", quoted.stdout.strip())

❌ files created by the unquoted command: ['=1.24'] | containing: pip install numpy
   → pip would install the LATEST numpy with no constraint, and hide its output in that file
✅ quoted, the command receives: pip install numpy>=1.24


### ❌ Pitfall 2 — A build backend that doesn't exist

Straight from the old version of this notebook. (`--no-isolation` uses the installed setuptools, so this needs no download.)

In [36]:
bad_backend = WORK / "bad-backend"
write(bad_backend / "pyproject.toml", '''
    [build-system]
    requires = ["setuptools>=61.0"]
    build-backend = "setuptools.backends.legacy:build"

    [project]
    name = "badpkg"
    version = "0.1.0"
''')
write(bad_backend / "src/badpkg/__init__.py", "")
print("❌")
run([sys.executable, "-m", "build", "--no-isolation", "--wheel", "--outdir", bad_backend / "dist", bad_backend], check=False, tail=True, max_lines=2)

good_backend = (bad_backend / "pyproject.toml").read_text().replace("setuptools.backends.legacy:build", "setuptools.build_meta")
(bad_backend / "pyproject.toml").write_text(good_backend)
print("\n✅")
run([sys.executable, "-m", "build", "--no-isolation", "--wheel", "--outdir", bad_backend / "dist", bad_backend], tail=True, max_lines=1);

❌


$ python -m build --no-isolation --wheel --outdir $WORK/bad-backend/dist $WORK/bad-backend
  … (8 earlier lines hidden)
  TIP pass --env-dir and --sdist-extract-dir to keep the build environment and sources, then see https://build.pypa.io/en/stable/how-to/troubleshooting.html#debug-a-failed-build for help debugging a failed build
  ERROR Backend 'setuptools.backends.legacy:build' is not available.
  [exit code 1]

✅


$ python -m build --no-isolation --wheel --outdir $WORK/bad-backend/dist $WORK/bad-backend
  … (39 earlier lines hidden)
  * Building wheel...


### ❌ Pitfall 3 — `pip install` goes into a different Python than your notebook

In [37]:
pip_on_path = shutil.which("pip")
print("notebook kernel runs:", short(sys.executable))
if pip_on_path is None:
    print("no `pip` command on PATH at all — `!pip install` would fail here")
else:
    pip_reports = run([pip_on_path, "--version"], show=False, check=False).stdout.strip()
    print("`pip` on PATH is    :", short(pip_reports))
    same_env = Path(pip_on_path).parent.resolve() == Path(sys.executable).parent.resolve()
    print("❌ `!pip install x` would install into a DIFFERENT environment than this kernel" if not same_env
          else "(on this machine they happen to match — don't rely on that)")
print("✅ target the kernel explicitly:  %pip install x   |   python -m pip install x   |   "
      f"uv pip install --python {short(sys.executable)} x")

notebook kernel runs: ~/Workspaces/Learning-AI-ML/.venv/bin/python


`pip` on PATH is    : pip 26.0.1 from /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip (python 3.13)
❌ `!pip install x` would install into a DIFFERENT environment than this kernel
✅ target the kernel explicitly:  %pip install x   |   python -m pip install x   |   uv pip install --python ~/Workspaces/Learning-AI-ML/.venv/bin/python x


### ❌ Pitfall 4 — Naming your file after a library

In [38]:
shadow = WORK / "shadow-demo"
write(shadow / "random.py", 'print("this is MY random.py, not the standard library!")\n')
print("❌ a script in a folder that contains random.py:")
run([sys.executable, "-c", "import random; print(random.randint(1, 6))"], cwd=shadow, check=False, tail=True, max_lines=2)

(shadow / "random.py").rename(shadow / "random_utils.py")
print("✅ after renaming it to random_utils.py:")
run([sys.executable, "-c", "import random; random.seed(0); print('dice roll:', random.randint(1, 6))"], cwd=shadow);

❌ a script in a folder that contains random.py:
$ python -c import random; print(random.randint(1, 6))
  … (2 earlier lines hidden)
    File "<string>", line 1, in <module>
  AttributeError: module 'random' has no attribute 'randint'
  [exit code 1]
✅ after renaming it to random_utils.py:
$ python -c import random; random.seed(0); print('dice roll:', random.randint(1, 6))
  dice roll: 4


### ❌ Pitfall 5 — Tests pass from the repo root, but the installed package is broken

A flat layout plus a hand-written package list forgets a subpackage. Importing from the project folder hides the bug; the wheel doesn't contain it.

In [39]:
flat = WORK / "flat-bug"
flat_config = '''
    [build-system]
    requires = ["setuptools>=77"]
    build-backend = "setuptools.build_meta"

    [project]
    name = "flatpkg"
    version = "0.1.0"

    [tool.setuptools]
    packages = ["flatpkg"]
'''
write(flat / "pyproject.toml", flat_config)
write(flat / "flatpkg/__init__.py", "")
write(flat / "flatpkg/io/__init__.py", "")
write(flat / "flatpkg/io/readers.py", "def read():\n    return 'data'\n")

run([sys.executable, "-c", "import flatpkg.io.readers as r; print('from the repo root it works:', r.read())"], cwd=flat)
run([sys.executable, "-m", "build", "--no-isolation", "--wheel", "--outdir", flat / "dist", flat], show=False)
flat_python = make_venv(WORK / "venv-flat-bug")
pip_install(flat_python, next((flat / "dist").glob("*.whl")))
print("❌ the installed wheel:")
run([flat_python, "-c", "import flatpkg.io.readers"], cwd=WORK, check=False, tail=True, max_lines=1)

shutil.rmtree(flat / "dist")
write(flat / "pyproject.toml", flat_config.replace('packages = ["flatpkg"]', "").replace("[tool.setuptools]", '[tool.setuptools.packages.find]\n    include = ["flatpkg*"]'))
run([sys.executable, "-m", "build", "--no-isolation", "--wheel", "--outdir", flat / "dist", flat], show=False)
fixed_python = make_venv(WORK / "venv-flat-fixed")
pip_install(fixed_python, next((flat / "dist").glob("*.whl")))
print("✅ with automatic package discovery:")
run([fixed_python, "-c", "import flatpkg.io.readers as r; print('installed package works:', r.read())"], cwd=WORK);

$ python -c import flatpkg.io.readers as r; print('from the repo root it works:', r.read())
  from the repo root it works: data


$ uv pip install --python $WORK/venv-flat-bug/bin/python $WORK/flat-bug/dist/flatpkg-0.1.0-py3-none-any.whl
  Using Python 3.12.11 environment at: $WORK/venv-flat-bug
  Resolved 1 package in 1ms
  Prepared 1 package in 3ms
  Installed 1 package in 2ms
   + flatpkg==0.1.0 (from file://$WORK/flat-bug/dist/flatpkg-0.1.0-py3-none-any.whl)
❌ the installed wheel:
$ $WORK/venv-flat-bug/bin/python -c import flatpkg.io.readers
  … (2 earlier lines hidden)
  ModuleNotFoundError: No module named 'flatpkg.io'
  [exit code 1]


$ uv pip install --python $WORK/venv-flat-fixed/bin/python $WORK/flat-bug/dist/flatpkg-0.1.0-py3-none-any.whl
  Using Python 3.12.11 environment at: $WORK/venv-flat-fixed
  Resolved 1 package in 1ms
  Prepared 1 package in 13ms
  Installed 1 package in 2ms
   + flatpkg==0.1.0 (from file://$WORK/flat-bug/dist/flatpkg-0.1.0-py3-none-any.whl)
✅ with automatic package discovery:
$ $WORK/venv-flat-fixed/bin/python -c import flatpkg.io.readers as r; print('installed package works:', r.read())
  installed package works: data


### ❌ Pitfall 6 — `pip freeze` as your dependency list

In [40]:
frozen = sorted({f"{d.metadata['Name']}=={d.version}" for d in md.distributions()}, key=str.lower)
print(f"❌ `pip freeze` in this notebook's environment would list {len(frozen)} packages, e.g. {frozen[:4]}")
print("   — every experiment ever installed here, with pins that may not exist on another OS")
print('✅ declare what your code imports, e.g. dependencies = ["numpy>=2.0", "scikit-learn>=1.5"], then let uv/pip-tools lock the rest')

❌ `pip freeze` in this notebook's environment would list 382 packages, e.g. ['absl-py==2.5.0', 'accelerate==1.15.0', 'aiohappyeyeballs==2.7.1', 'aiohttp-cors==0.8.1']
   — every experiment ever installed here, with pins that may not exist on another OS
✅ declare what your code imports, e.g. dependencies = ["numpy>=2.0", "scikit-learn>=1.5"], then let uv/pip-tools lock the rest


## 🏋️ Practice Exercises

Try each one before opening the solution. Run the cell: ⏳ means not attempted, ✅ means correct.

### 🟢 Exercise 1 — Filter allowed versions
Keep the versions in `available` that satisfy **`>=2.0,<2.3` and are not `2.1.1`** (a known-broken release). Preserve order.

In [41]:
available = ["1.26.4", "2.0.0", "2.1.0", "2.1.1", "2.2.0", "2.3.0rc1", "2.3.0"]
allowed = None  # TODO
check("allowed", allowed, ["2.0.0", "2.1.0", "2.2.0"], hint='One SpecifierSet can hold all three clauses separated by commas.')

⏳ allowed: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
available = ["1.26.4", "2.0.0", "2.1.0", "2.1.1", "2.2.0", "2.3.0rc1", "2.3.0"]
allowed = list(SpecifierSet(">=2.0,<2.3,!=2.1.1").filter(available))
check("allowed", allowed, ["2.0.0", "2.1.0", "2.2.0"])
```
</details>

### 🟢 Exercise 2 — Parse a `.env` file yourself
Write `parse_env(text)` returning a dict: skip blank lines and `#` comments, split each line on the **first** `=`, strip whitespace and surrounding double quotes from values. (python-dotenv gives the same result for this input.)

In [42]:
env_text = '# local settings\nDB_HOST=localhost\n\nAPI_KEY="abc123"\nDEBUG=true\nEMPTY=\n'


def parse_env(text):
    return None  # TODO


check("parse_env", parse_env(env_text), {"DB_HOST": "localhost", "API_KEY": "abc123", "DEBUG": "true", "EMPTY": ""},
      hint="line.split('=', 1) and value.strip().strip('\"')")

⏳ parse_env: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
env_text = '# local settings\nDB_HOST=localhost\n\nAPI_KEY="abc123"\nDEBUG=true\nEMPTY=\n'


def parse_env(text):
    result = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, _, value = line.partition("=")
        result[key.strip()] = value.strip().strip('"')
    return result


expected = {"DB_HOST": "localhost", "API_KEY": "abc123", "DEBUG": "true", "EMPTY": ""}
from dotenv import dotenv_values
assert dict(dotenv_values(stream=io.StringIO(env_text))) == expected      # the real library agrees
check("parse_env", parse_env(env_text), expected)
```
Real `.env` parsers also handle `export KEY=...`, single quotes, escapes, and inline comments — use python-dotenv in production.
</details>

### 🟢 Exercise 3 — Resolve an entry point
`[project.scripts]` values look like `"package.module:function"`. Write `load_entry_point(spec)` that imports the module and returns the attribute. The check uses standard-library targets.

In [43]:
def load_entry_point(spec):
    return None  # TODO


got = None
if load_entry_point("json:dumps") is not None:
    got = [load_entry_point("json:dumps")({"lr": 0.001}), load_entry_point("os.path:basename")("/data/train.csv")]
check("entry_point", got, ['{"lr": 0.001}', "train.csv"], hint="spec.partition(':') then importlib.import_module and getattr.")

⏳ entry_point: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def load_entry_point(spec):
    module_name, _, attribute_path = spec.partition(":")
    obj = importlib.import_module(module_name)
    for part in attribute_path.split("."):          # also supports "module:Class.method"
        obj = getattr(obj, part)
    return obj


got = [load_entry_point("json:dumps")({"lr": 0.001}), load_entry_point("os.path:basename")("/data/train.csv")]
check("entry_point", got, ['{"lr": 0.001}', "train.csv"])
```
This is essentially what the generated console-script launcher does before calling `sys.exit(main())`.
</details>

### 🟡 Exercise 4 — Which requirements apply on a Linux GPU server?
Return the **names** of requirements in `requirements` that would be installed on `linux_gpu_box` (keep order). Use `packaging.requirements.Requirement` and marker evaluation.

In [44]:
requirements = [
    "numpy>=2.0",
    'tomli>=2.0 ; python_version < "3.11"',
    'pywin32>=306 ; sys_platform == "win32"',
    'nvidia-cudnn-cu12 ; sys_platform == "linux" and platform_machine == "x86_64"',
    'uvloop ; sys_platform != "win32"',
]
linux_gpu_box = {"sys_platform": "linux", "platform_machine": "x86_64", "python_version": "3.12"}
needed = None  # TODO
check("needed", needed, ["numpy", "nvidia-cudnn-cu12", "uvloop"],
      hint="A requirement applies if req.marker is None or req.marker.evaluate(linux_gpu_box).")

⏳ needed: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
requirements = [
    "numpy>=2.0",
    'tomli>=2.0 ; python_version < "3.11"',
    'pywin32>=306 ; sys_platform == "win32"',
    'nvidia-cudnn-cu12 ; sys_platform == "linux" and platform_machine == "x86_64"',
    'uvloop ; sys_platform != "win32"',
]
linux_gpu_box = {"sys_platform": "linux", "platform_machine": "x86_64", "python_version": "3.12"}
parsed = [Requirement(line) for line in requirements]
needed = [r.name for r in parsed if r.marker is None or r.marker.evaluate(linux_gpu_box)]
check("needed", needed, ["numpy", "nvidia-cudnn-cu12", "uvloop"])
```
`evaluate()` fills in any variable you don't pass (like `implementation_name`) from the *current* machine — that's how lock tools compute per-platform installs.
</details>

### 🟡 Exercise 5 — Package discovery from scratch
Implement `find_regular_packages(where)`: return the sorted dotted names of every directory under `where` that contains `__init__.py` **and whose parents (inside `where`) are packages too**. It must match `setuptools.find_packages`.

In [45]:
discovery_root = WORK / "discovery"
for relative in ["core/__init__.py", "core/models/__init__.py", "core/models/linear.py", "core/utils/helpers.py",
                 "core/utils/io/__init__.py", "plugins/__init__.py", "scripts/run.py"]:
    write(discovery_root / relative, "")

from setuptools import find_packages

setuptools_answer = sorted(find_packages(where=str(discovery_root)))
print("setuptools finds:", setuptools_answer, "← core/utils has no __init__.py, so core.utils.io is skipped too")

setuptools finds: ['core', 'core.models', 'plugins'] ← core/utils has no __init__.py, so core.utils.io is skipped too


In [46]:
def find_regular_packages(where):
    return None  # TODO


check("find_regular_packages", find_regular_packages(discovery_root), setuptools_answer,
      hint="Recurse: a directory counts only if it has __init__.py; only descend into directories that count.")

⏳ find_regular_packages: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
def find_regular_packages(where):
    found = []

    def walk(directory, prefix):
        for child in sorted(p for p in directory.iterdir() if p.is_dir()):
            if (child / "__init__.py").exists():
                name = f"{prefix}{child.name}"
                found.append(name)
                walk(child, name + ".")

    walk(Path(where), "")
    return sorted(found)


check("find_regular_packages", find_regular_packages(discovery_root), setuptools_answer)
```
`setuptools.find_namespace_packages` is the PEP 420 variant that also accepts directories without `__init__.py`.
</details>

### 🔴 Exercise 6 — Sort versions correctly (PEP 440 subset)
Write `version_key(v)` for versions made of dotted numbers with an optional pre-release suffix `aN`, `bN`, or `rcN` (e.g. `1.10.0rc1`). Sorting with it must match `packaging.version.Version`. Remember: `1.10 == 1.10.0`, and a pre-release sorts **before** its final release.

In [47]:
versions_to_sort = ["1.10.0", "1.2.0", "1.10.0rc1", "1.10.0a2", "1.10.0b1", "1.9.9", "1.10", "2.0.0rc2", "1.10.1", "0.9"]


def version_key(v):
    return None  # TODO


got = None if version_key("1.0") is None else sorted(versions_to_sort, key=version_key)
check("version_key", got, sorted(versions_to_sort, key=Version),
      hint="Key = (release numbers with trailing zeros removed, (pre-release rank, number)); a final release gets a rank above 'rc'.")

⏳ version_key: not attempted yet — replace None with your answer.


<details><summary>💡 Show solution</summary>

```python
versions_to_sort = ["1.10.0", "1.2.0", "1.10.0rc1", "1.10.0a2", "1.10.0b1", "1.9.9", "1.10", "2.0.0rc2", "1.10.1", "0.9"]


def version_key(v):
    match = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    if match is None:
        raise ValueError(f"unsupported version: {v}")
    release = [int(part) for part in match.group(1).split(".")]
    while len(release) > 1 and release[-1] == 0:      # 1.10.0 == 1.10
        release.pop()
    if match.group(2):
        pre = (["a", "b", "rc"].index(match.group(2)), int(match.group(3)))
    else:
        pre = (3, 0)                                   # final release sorts after a, b, rc
    return tuple(release), pre


got = sorted(versions_to_sort, key=version_key)
check("version_key", got, sorted(versions_to_sort, key=Version))
```

Talking points: tuples compare element by element, so `(1, 9, 9) < (1, 10)`; equal versions keep their input order because Python's sort is stable. Full PEP 440 adds epochs (`1!2.0`), post-releases, dev releases, and local versions (`+cu121`).
</details>

## 🚀 Mini Project: Package a Text Statistics CLI

**Goal:** turn a useful utility into a real, installable package: `textstats FILE` prints word, sentence and vocabulary statistics. We'll use a `src/` layout, `pyproject.toml`, tests, lint, build a wheel, install it into a **fresh** environment, run the tests against the installed package, and analyse a real book — *Alice's Adventures in Wonderland* from Project Gutenberg — verifying the CLI's numbers independently.

**Steps:** scaffold → write the code → write tests → lint → build → install & test → run on real data.

### Step 1 — Scaffold the project

In [48]:
project = WORK / "textstats"
write(project / "pyproject.toml", '''
    [build-system]
    requires = ["setuptools>=77"]
    build-backend = "setuptools.build_meta"

    [project]
    name = "textstats"
    version = "0.1.0"
    description = "Word, sentence and vocabulary statistics for plain-text files"
    readme = "README.md"
    requires-python = ">=3.10"
    license = "MIT"
    dependencies = []                       # standard library only

    [project.scripts]
    textstats = "textstats.cli:main"

    [dependency-groups]
    dev = ["pytest>=8", "ruff>=0.6"]

    [tool.pytest.ini_options]
    testpaths = ["tests"]
    addopts = "-q"

    [tool.ruff]
    line-length = 100
    target-version = "py310"

    [tool.ruff.lint]
    select = ["E", "F", "I", "B", "UP"]
''')
write(project / "README.md", '''
    # textstats

    ```bash
    textstats book.txt --top 10 [--json] [--gutenberg]
    ```
''')
write(project / ".gitignore", '''
    .venv/
    __pycache__/
    *.egg-info/
    build/
    dist/
    .pytest_cache/
    .ruff_cache/
    .env
''')
print("scaffolded", short(project))

scaffolded $WORK/textstats


### Step 2 — Write the code

Keep **pure logic** (`core.py`: no printing, no files — easy to test) separate from the **interface** (`cli.py`: arguments and output).

In [49]:
write(project / "src/textstats/__init__.py", '''
    """textstats — word, sentence and vocabulary statistics for plain text."""
    from importlib.metadata import PackageNotFoundError, version

    try:
        __version__ = version("textstats")
    except PackageNotFoundError:  # running from an uninstalled source checkout
        __version__ = "0.0.0+unknown"

    from textstats.core import TextStats, analyze, tokenize  # noqa: E402

    __all__ = ["TextStats", "__version__", "analyze", "tokenize"]
''')
write(project / "src/textstats/core.py", '''
    """Pure functions: no printing and no file I/O, so they are easy to test."""
    import re
    from collections import Counter
    from dataclasses import dataclass

    WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)*")
    SENTENCE_END_RE = re.compile(r"[.!?]+(?=\\s|$)")


    @dataclass(frozen=True)
    class TextStats:
        n_characters: int
        n_words: int
        n_unique_words: int
        n_sentences: int
        avg_word_length: float
        reading_minutes: float
        top_words: tuple[tuple[str, int], ...]


    def tokenize(text: str) -> list[str]:
        """Lower-case words; curly apostrophes are normalised so "Alice’s" == "alice's"."""
        return WORD_RE.findall(text.lower().replace("\\u2019", "'"))


    def strip_gutenberg_boilerplate(text: str) -> str:
        """Keep only the book between Project Gutenberg's START and END markers."""
        start, end = text.find("*** START OF"), text.find("*** END OF")
        if start == -1 or end == -1:
            return text
        return text[text.index("\\n", start) + 1 : end]


    def analyze(text: str, top: int = 10, words_per_minute: int = 238) -> TextStats:
        words = tokenize(text)
        counts = Counter(words)
        top_words = tuple(sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:top])
        return TextStats(
            n_characters=len(text),
            n_words=len(words),
            n_unique_words=len(counts),
            n_sentences=len(SENTENCE_END_RE.findall(text)),
            avg_word_length=round(sum(map(len, words)) / len(words), 3) if words else 0.0,
            reading_minutes=round(len(words) / words_per_minute, 1),
            top_words=top_words,
        )
''')
write(project / "src/textstats/cli.py", '''
    """Command-line interface: argument parsing and printing only."""
    import argparse
    import json
    import sys
    from dataclasses import asdict
    from pathlib import Path

    from textstats import __version__
    from textstats.core import analyze, strip_gutenberg_boilerplate


    def build_parser() -> argparse.ArgumentParser:
        parser = argparse.ArgumentParser(prog="textstats", description="Word and sentence statistics.")
        parser.add_argument("path", help="text file to analyse, or - for standard input")
        parser.add_argument("--top", type=int, default=10, help="number of frequent words to show")
        parser.add_argument("--json", action="store_true", help="print machine-readable JSON")
        parser.add_argument("--gutenberg", action="store_true", help="strip Gutenberg header/footer")
        parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
        return parser


    def main(argv: list[str] | None = None) -> int:
        args = build_parser().parse_args(argv)
        try:
            text = sys.stdin.read() if args.path == "-" else Path(args.path).read_text(encoding="utf-8")
        except OSError as err:
            print(f"textstats: error: {err}", file=sys.stderr)
            return 1
        if args.gutenberg:
            text = strip_gutenberg_boilerplate(text)
        stats = analyze(text, top=args.top)
        if args.json:
            print(json.dumps(asdict(stats), indent=2))
            return 0
        print(f"words: {stats.n_words:,} | unique: {stats.n_unique_words:,}")
        print(f"sentences: {stats.n_sentences:,} | average word length: {stats.avg_word_length}")
        print(f"reading time: {stats.reading_minutes} min")
        for word, count in stats.top_words:
            print(f"  {word:<12} {count:>6,}")
        return 0
''')
write(project / "src/textstats/__main__.py", '''
    from textstats.cli import main

    raise SystemExit(main())
''')
tree(project)

📁 textstats/
    📄 .gitignore
    📄 README.md
    📄 pyproject.toml
    📁 src/
        📁 textstats/
            📄 __init__.py
            📄 __main__.py
            📄 cli.py
            📄 core.py


### Step 3 — Write the tests

In [50]:
write(project / "tests/test_core.py", '''
    from textstats import analyze, tokenize
    from textstats.core import strip_gutenberg_boilerplate


    def test_tokenize_lowercases_and_keeps_apostrophes():
        assert tokenize("Don't PANIC, Alice\\u2019s cat!") == ["don't", "panic", "alice's", "cat"]


    def test_analyze_counts_words_sentences_and_top_words():
        stats = analyze("The cat sat. The cat ran! Did it?", top=2)
        assert (stats.n_words, stats.n_unique_words, stats.n_sentences) == (8, 6, 3)
        assert stats.top_words == (("cat", 2), ("the", 2))


    def test_empty_text():
        stats = analyze("")
        assert stats.n_words == 0 and stats.avg_word_length == 0.0 and stats.top_words == ()


    def test_strip_gutenberg_boilerplate():
        raw = "header\\n*** START OF THE BOOK ***\\nreal text\\n*** END OF THE BOOK ***\\nlicense"
        assert strip_gutenberg_boilerplate(raw).strip() == "real text"
''')
write(project / "tests/test_cli.py", '''
    import json

    import pytest

    from textstats.cli import main


    def test_json_output(tmp_path, capsys):
        sample = tmp_path / "sample.txt"
        sample.write_text("Hello world. Hello again!", encoding="utf-8")
        assert main([str(sample), "--json", "--top", "1"]) == 0
        result = json.loads(capsys.readouterr().out)
        assert result["n_words"] == 4
        assert result["top_words"] == [["hello", 2]]


    def test_missing_file_returns_error_code(capsys):
        assert main(["does-not-exist.txt"]) == 1
        assert "error" in capsys.readouterr().err


    @pytest.mark.parametrize("flag", ["--help", "--version"])
    def test_help_and_version_exit_cleanly(flag):
        with pytest.raises(SystemExit) as exc:
            main([flag])
        assert exc.value.code == 0
''')
print("tests:", sorted(p.name for p in (project / "tests").glob("test_*.py")))

tests: ['test_cli.py', 'test_core.py']


### Step 4 — Lint

In [51]:
if not UV:
    print("⏭️ Skipped: needs uv (`uvx ruff check .`) or `pip install ruff`.")
else:
    lint = run([UV, "tool", "run", "ruff", "check", "--output-format", "concise", "."], cwd=project, check=False)
    print("✅ lint clean" if lint.returncode == 0 else "❌ fix the issues above")

$ uv tool run ruff check --output-format concise .
  All checks passed!
✅ lint clean


### Step 5 — Build the wheel and sdist

In [52]:
start = time.perf_counter()
built = run([sys.executable, "-m", "build", "--outdir", project / "dist", project], check=False, tail=True, max_lines=1)
if built.returncode != 0:
    print("⚠️ The isolated build needs PyPI access; retrying with --no-isolation (installed setuptools).")
    run([sys.executable, "-m", "build", "--no-isolation", "--outdir", project / "dist", project], tail=True, max_lines=1)
textstats_wheel = next((project / "dist").glob("*.whl"))
print(f"built in {time.perf_counter() - start:.1f} s → {sorted(p.name for p in (project / 'dist').iterdir())}")
with zipfile.ZipFile(textstats_wheel) as zf:
    print("wheel ships:", [n for n in zf.namelist() if n.startswith("textstats/")])

$ python -m build --outdir $WORK/textstats/dist $WORK/textstats
  … (101 earlier lines hidden)
  * Building wheel...
built in 5.1 s → ['textstats-0.1.0-py3-none-any.whl', 'textstats-0.1.0.tar.gz']
wheel ships: ['textstats/__init__.py', 'textstats/__main__.py', 'textstats/cli.py', 'textstats/core.py']


### Step 6 — Install into a fresh environment and test the *installed* package

In [53]:
ts_venv = WORK / "venv-textstats"
ts_python = make_venv(ts_venv)
installed = pip_install(ts_python, textstats_wheel, "pytest>=8", check=False)
if installed.returncode != 0:
    print("⏭️ Skipped: could not install pytest from PyPI (offline?). Installing only the wheel:")
    pip_install(ts_python, textstats_wheel)
else:
    tests = run([ts_python, "-m", "pytest", project / "tests"], cwd=WORK)
    passed = re.search(r"(\d+) passed", tests.stdout)
    print(f"✅ {passed.group(1)} tests passed against the installed wheel" if passed else "❌ tests did not pass")

textstats_cmd = venv_bin(ts_venv, "textstats")
run([textstats_cmd, "--version"])
run([ts_python, "-m", "textstats", "--help"], max_lines=4);

$ uv pip install --python $WORK/venv-textstats/bin/python $WORK/textstats/dist/textstats-0.1.0-py3-none-any.whl pytest>=8
  Using Python 3.12.11 environment at: $WORK/venv-textstats
  Resolved 6 packages in 8ms
  Prepared 1 package in 3ms
  Installed 6 packages in 12ms
   + iniconfig==2.3.0
   + packaging==26.3
   + pluggy==1.6.0
   + pygments==2.21.0
   + pytest==9.1.1
   + textstats==0.1.0 (from file://$WORK/textstats/dist/textstats-0.1.0-py3-none-any.whl)


$ $WORK/venv-textstats/bin/python -m pytest $WORK/textstats/tests
  ........                                                                 [100%]
  8 passed in 0.03s
✅ 8 tests passed against the installed wheel


$ $WORK/venv-textstats/bin/textstats --version
  textstats 0.1.0
$ $WORK/venv-textstats/bin/python -m textstats --help
  usage: textstats [-h] [--top TOP] [--json] [--gutenberg] [--version] path
  
  Word and sentence statistics.
  
  … (9 more lines)


### Step 7 — Run it on a real book and verify the numbers

In [54]:
alice_path = OUTPUT_DIR / "alice_in_wonderland.txt"
alice_url = "https://www.gutenberg.org/cache/epub/11/pg11.txt"
if not alice_path.exists():
    try:
        with urllib.request.urlopen(urllib.request.Request(alice_url, headers={"User-Agent": "course-notebook"}), timeout=30) as response:
            alice_path.write_bytes(response.read())
    except OSError as err:
        print(f"⏭️ Could not download the book ({err}); Step 7 needs network access once.")

if alice_path.exists():
    print(f"book: {alice_path.stat().st_size / 1024:.0f} KB from Project Gutenberg (public domain)\n")
    start = time.perf_counter()
    run([textstats_cmd, alice_path, "--gutenberg", "--top", "8"])
    print(f"(CLI finished in {time.perf_counter() - start:.2f} s)")

    report = json.loads(run([textstats_cmd, alice_path, "--gutenberg", "--json", "--top", "25"], show=False).stdout)

    # Independent check, written without importing the package
    raw = alice_path.read_text(encoding="utf-8")
    body = raw[raw.index("\n", raw.find("*** START OF")) + 1 : raw.find("*** END OF")]
    words = re.findall(r"[a-z]+(?:'[a-z]+)*", body.lower().replace("’", "'"))
    expected_top = sorted(Counter(words).items(), key=lambda kv: (-kv[1], kv[0]))[:25]
    assert report["n_words"] == len(words)
    assert report["n_unique_words"] == len(set(words))
    assert [tuple(pair) for pair in report["top_words"]] == expected_top
    print(f"\n✅ CLI matches an independent count: {len(words):,} words, {len(set(words)):,} distinct")

    top_ranked = [w for w, _ in report["top_words"]]
    rank = top_ranked.index("alice") + 1 if "alice" in top_ranked else None
    print(f"'alice' is the #{rank} most frequent word" if rank else "'alice' is not in the top 25 words")
    print(f"share of the text taken by the 25 most common words: {sum(c for _, c in report['top_words']) / report['n_words']:.1%}")

book: 170 KB from Project Gutenberg (public domain)



$ $WORK/venv-textstats/bin/textstats _outputs/alice_in_wonderland.txt --gutenberg --top 8
  words: 26,776 | unique: 2,635
  sentences: 967 | average word length: 4.061
  reading time: 112.5 min
    the           1,651
    and             874
    to              729
    a               637
    she             541
    it              530
    of              515
    said            462
(CLI finished in 0.16 s)



✅ CLI matches an independent count: 26,776 words, 2,635 distinct
'alice' is the #10 most frequent word
share of the text taken by the 25 most common words: 38.0%


**Stretch goals**
1. Add a `--stopwords` flag that removes common words (`the`, `and`, …) before counting, with a test.
2. Publish to **TestPyPI** with `uv publish --publish-url https://test.pypi.org/legacy/` (needs an account and API token) and install it from there.
3. Add a GitHub Actions workflow that runs `uv sync --locked`, `ruff check`, and `pytest` on Linux, macOS and Windows.

### 🗣️ How to talk about this in an interview
- "I packaged a CLI with a `src/` layout and a PEP 621 `pyproject.toml`: setuptools build backend, a console-script entry point, and dev tools in a dependency group."
- "I separated pure logic from the CLI layer, so unit tests don't touch files, and used pytest fixtures (`tmp_path`, `capsys`) for the I/O parts."
- "I built a wheel, installed it into a clean venv, and ran the tests against the *installed* package — that catches missing modules that tests from the repo root would hide."
- "I validated the tool on a real book by comparing its JSON output with an independent word count."

## 🎤 Interview Q&A

Try answering **out loud** before opening each answer.

### 🧠 Concepts

**Q1. What *is* a virtual environment, physically?**

<details><summary>Show answer</summary>

- **30-second answer:** A directory containing `pyvenv.cfg`, a `bin/` (or `Scripts\`) folder with a `python` that points at a base interpreter, and its own `site-packages`. When that `python` starts it reads `pyvenv.cfg`, sets `sys.prefix` to the venv, and imports/installs go to the venv's site-packages while the standard library comes from the base Python.
- **Go deeper:** Detect it with `sys.prefix != sys.base_prefix`. `activate` only edits `PATH` and sets `VIRTUAL_ENV`, so `.venv/bin/python app.py` works without it. Venvs aren't relocatable — recreate rather than copy them. conda environments additionally hold non-Python binaries.
- **❌ Common wrong answer:** "It's a full copy of Python" or "it only works after you activate it."

</details>

**Q2. `pyproject.toml` dependencies vs `requirements.txt` vs a lock file?**

<details><summary>Show answer</summary>

- **30-second answer:** `[project] dependencies` are abstract ranges describing what the project needs — right for libraries. A lock file (`uv.lock`, `pylock.toml`, `poetry.lock`, or a compiled requirements file) records exact versions and hashes of every transitive dependency — right for applications, CI and serving. A hand-written `requirements.txt` sits in between; `pip freeze` just dumps whatever is installed.
- **Go deeper:** Libraries shouldn't pin exact versions (it causes conflicts for their users). PEP 751 standardised `pylock.toml`. Install locks in CI with `uv sync --locked`; upgrade deliberately with `uv lock --upgrade-package <name>`.
- **❌ Common wrong answer:** "`pip freeze > requirements.txt` is a proper lock file" or "pin everything with `==` in a library."

</details>

**Q3. What's the difference between a wheel and an sdist?**

<details><summary>Show answer</summary>

- **30-second answer:** An sdist is a source archive that has to be built on the installing machine (possibly needing a compiler). A wheel is a pre-built zip that installers just unpack — fast and reproducible. Wheel filenames carry tags: `py3-none-any` is pure Python; `cp312-cp312-manylinux_2_28_x86_64` is compiled for one interpreter and platform.
- **Go deeper:** Inside a wheel, `*.dist-info/` holds `METADATA`, `WHEEL`, `RECORD` (hashes) and `entry_points.txt`. Projects with C extensions build many platform wheels in CI (e.g. with cibuildwheel). Publish both an sdist and wheels.
- **❌ Common wrong answer:** "A wheel is compiled Python bytecode."

</details>

**Q4. What goes into `pyproject.toml`?**

<details><summary>Show answer</summary>

- **30-second answer:** `[build-system]` names the build backend and its requirements (PEP 517/518); `[project]` holds standard metadata — name, version, `requires-python`, dependencies, optional dependencies, scripts (PEP 621); `[dependency-groups]` holds dev-only groups (PEP 735); `[tool.*]` configures tools such as pytest, ruff, uv, or setuptools.
- **Go deeper:** Common backends: `setuptools.build_meta`, `hatchling.build`, `uv_build`, `flit_core.buildapi`, `poetry.core.masonry.api`. Frontends (pip, uv, build) create an isolated build environment from `requires`. `setup.py` is only needed for unusual custom build logic.
- **❌ Common wrong answer:** "`pyproject.toml` is Poetry's config file" or a made-up backend like `setuptools.backends.legacy:build`.

</details>

**Q5. Does a directory need `__init__.py` to be importable?**

<details><summary>Show answer</summary>

- **30-second answer:** No. Since Python 3.3 (PEP 420), a directory without `__init__.py` on `sys.path` imports as a **namespace package**; it can span several directories and has `__file__ = None`. Regular packages (with `__init__.py`) live in one place and run `__init__.py` on import.
- **Go deeper:** Namespace packages suit plugin families distributed separately (`acme.vision`, `acme.nlp`). `setuptools.find_packages` ignores directories without `__init__.py` (use `find_namespace_packages`), which is a common source of "module missing from the wheel" bugs.
- **❌ Common wrong answer:** "Yes, otherwise Python can't import it."

</details>

**Q6. Why do many projects use a `src/` layout?**

<details><summary>Show answer</summary>

- **30-second answer:** With the package under `src/`, it isn't importable from the project root by accident, so tests and scripts must use the installed package. Packaging mistakes — missing modules, data files, dependencies — then show up locally instead of for your users.
- **Go deeper:** Use an editable install during development. A flat layout is fine for small applications that are never distributed. `uv init --package` and `uv init --lib` generate a `src/` layout.
- **❌ Common wrong answer:** "It's purely a style preference."

</details>

**Q7. uv, pip, Poetry, or conda — how do you choose?**

<details><summary>Show answer</summary>

- **30-second answer:** uv is a fast, standards-based all-in-one tool (Python versions, venvs, locking, syncing, running, tools) and a good default for new projects. pip + venv is the universal baseline. Poetry is a mature project manager with its own lock file. conda (Miniforge, mamba, pixi) is the choice when you need non-Python binaries such as CUDA toolkits or GDAL from conda-forge.
- **Go deeper:** In conda environments, install conda packages first and pip packages last. Anaconda's `defaults` channel requires a paid plan for for-profit organisations over 200 employees; conda-forge is community-run. In production, Docker usually handles system libraries.
- **❌ Common wrong answer:** "Data science requires conda" or "all lock files are interchangeable."

</details>

### 💻 Coding

**Q8. How do you make a command like `textstats` available after `pip install`?**

<details><summary>Show answer</summary>

- **30-second answer:** Declare `[project.scripts] textstats = "textstats.cli:main"`. On install, the installer generates a launcher in the environment's `bin/` that imports `main` and calls `sys.exit(main())`, so `main`'s return value becomes the exit code. Add `__main__.py` to support `python -m textstats`.
- **Go deeper:** The mapping lands in `entry_points.txt` and can be read with `importlib.metadata.entry_points(group="console_scripts")`. Custom groups (`[project.entry-points."myapp.plugins"]`) are how plugin systems discover code across packages.
- **❌ Common wrong answer:** "Copy the script into `/usr/local/bin` and `chmod +x` it."

</details>

**Q9. Explain `~=`, `==2.*`, extras, and markers — and why you quote them in a shell.**

<details><summary>Show answer</summary>

- **30-second answer:** `~=2.2` means `>=2.2, ==2.*`; `~=2.2.1` means `>=2.2.1, ==2.2.*`; `==2.*` is a prefix match. `pkg[extra]` installs optional dependencies; `; sys_platform == "linux"` makes a requirement conditional. Pre-releases are excluded unless requested. Quote because `>`/`<` are shell redirections: `pip install numpy>=1.24` installs the latest NumPy and writes output to a file named `=1.24`.
- **Go deeper:** Tools parse versions (PEP 440) because `"1.10" > "1.9"` is `False` as strings. Markers are evaluated per target environment by lock resolvers.
- **❌ Common wrong answer:** "`~=2.2` allows only patch releases."

</details>

### 🐛 Debugging Scenarios

**Q10. `pip install` succeeded, but `import` fails in your Jupyter notebook. Why?**

<details><summary>Show answer</summary>

- **30-second answer:** `pip` belonged to a different interpreter than the kernel. Compare `sys.executable` in the notebook with `pip --version`. Install with `%pip install …` (targets the running kernel), `python -m pip` using the kernel's Python, or register your venv as a kernel with `python -m ipykernel install --user --name myenv`.
- **Go deeper:** `!pip` runs whichever `pip` is first on `PATH`. With uv: `uv pip install --python <kernel python> …`. Restart the kernel after upgrading packages that were already imported.
- **❌ Common wrong answer:** "Reinstall Python" or "restart the computer."

</details>

**Q11. Tests passed yesterday; today CI fails without any code change. What happened and how do you prevent it?**

<details><summary>Show answer</summary>

- **30-second answer:** An unpinned (often transitive) dependency published a breaking release, and CI resolved it fresh. Commit a lock file and install with `uv sync --locked` (or compiled requirements with hashes), and upgrade dependencies deliberately in a reviewed PR.
- **Go deeper:** Pin the Python version too. Dependabot or Renovate can open lock-update PRs automatically. Compare `uv tree` or `pip freeze` output between the last green run and the failing one to find the culprit.
- **❌ Common wrong answer:** "Pin exact versions in the library's `pyproject.toml`."

</details>

**Q12. `torch.cuda.is_available()` returns `False` on a machine with an NVIDIA GPU.**

<details><summary>Show answer</summary>

- **30-second answer:** Check `torch.version.cuda`: `None` means a CPU-only build was installed (for example from the CPU index, or a platform whose default wheels are CPU-only). Install the CUDA build from PyTorch's index (e.g. `https://download.pytorch.org/whl/cu130`, or uv sources with markers) and check that the NVIDIA driver (`nvidia-smi`) supports that CUDA version.
- **Go deeper:** PyTorch pip wheels bundle the CUDA runtime libraries, so you need a compatible driver rather than a separately installed CUDA toolkit. Lock per-platform variants with markers, and build GPU Docker images on a matching CUDA base image.
- **❌ Common wrong answer:** "Reinstall the NVIDIA driver" before checking which torch build is installed.

</details>

**Q13. Everything imports fine in local tests, but users get `ModuleNotFoundError` after installing your wheel.**

<details><summary>Show answer</summary>

- **30-second answer:** Local tests imported the source folder from the repo root (flat layout), hiding that the wheel is missing a subpackage — for example a hand-written `packages = [...]` list or a directory without `__init__.py` skipped by `find_packages`. Inspect the wheel (`unzip -l dist/*.whl`), switch to a `src/` layout with automatic discovery, and run tests against the installed wheel in CI.
- **Go deeper:** Also check package data (non-`.py` files) and that runtime imports are declared in `dependencies`, not just installed in your dev environment.
- **❌ Common wrong answer:** "Add `sys.path.append(...)` in the package."

</details>

### 🏗️ Design

**Q14. Design a reproducible Python environment setup for an ML team, from laptop to production.**

<details><summary>Show answer</summary>

- **30-second answer:** One `pyproject.toml` with direct dependencies and dependency groups (dev, training, serving); a committed lock file and `.python-version`; CI running `uv sync --locked`, ruff and pytest; production Docker images built from the same lock on a pinned (CUDA) base image; secrets from environment variables or a secret manager; and an environment fingerprint logged with every trained model.
- **Go deeper:** Use markers and per-platform indexes for CPU vs CUDA PyTorch; keep serving images small with a separate group; automate lock updates; host internal packages on a private index.
- **❌ Common wrong answer:** "Everyone runs `pip install -r requirements.txt` from a wiki page."

</details>

**Q15. How do you handle API keys and other secrets in a Python ML project?**

<details><summary>Show answer</summary>

- **30-second answer:** Never in code or git. Read them from environment variables; locally load a git-ignored `.env` (python-dotenv or pydantic-settings) and commit a `.env.example` template; in production use the platform's secret manager. If a key leaks, rotate it immediately — removing the commit isn't enough.
- **Go deeper:** Add secret scanning (gitleaks, detect-secrets, GitHub push protection), mask secrets in logs, use separate least-privilege keys per environment, and clear notebook outputs that might display them.
- **❌ Common wrong answer:** "Put the key in `config.py` and add it to `.gitignore` later."

</details>

## 🧪 Quick Quiz

Predict the output, then reveal.

**1.** `print(Version("1.10") > Version("1.9"), "1.10" > "1.9")`
<details><summary>Answer</summary>

`True False` — version-aware comparison vs character-by-character string comparison.
</details>

**2.** `print("2.0rc1" in SpecifierSet(">=1.0"))`
<details><summary>Answer</summary>

`False` — pre-releases are excluded unless the specifier itself mentions a pre-release (or you allow them explicitly).
</details>

**3.** `list(SpecifierSet("~=2.2").filter(["2.1.9", "2.2.0", "2.9.1", "3.0.0"]))`
<details><summary>Answer</summary>

`['2.2.0', '2.9.1']` — `~=2.2` means `>=2.2, ==2.*`.
</details>

**4.** A folder `geo/` containing only `one.py` is on `sys.path`. What does `import geo; print(geo.__file__)` print?
<details><summary>Answer</summary>

`None` — `geo` is a PEP 420 namespace package, which has no `__init__.py` file behind it.
</details>

**5.** In bash you run `pip install numpy>=1.24`. What happens?
<details><summary>Answer</summary>

The shell runs `pip install numpy` (latest version, no constraint) and redirects its output into a new file named `=1.24`. Quote it: `pip install "numpy>=1.24"`.
</details>

## 📚 Resources

### 📖 Official Docs
- [uv](https://docs.astral.sh/uv/) — the official uv documentation · [Working on projects | uv](https://docs.astral.sh/uv/guides/projects/) · [Locking and syncing | uv](https://docs.astral.sh/uv/concepts/projects/sync/)
- [Using uv with PyTorch | uv](https://docs.astral.sh/uv/guides/integration/pytorch/) — CPU vs CUDA indexes and markers
- [Packaging Python Projects - Python Packaging User Guide](https://packaging.python.org/en/latest/tutorials/packaging-projects/) — the official end-to-end tutorial
- [Writing your pyproject.toml - Python Packaging User Guide](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/) · [src layout vs flat layout](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/)
- [venv — Creation of virtual environments](https://docs.python.org/3/library/venv.html) — how venvs work under the hood
- [Get Started - pytest documentation](https://docs.pytest.org/en/stable/getting-started.html) · [Configuring Ruff](https://docs.astral.sh/ruff/configuration/)
- [pip-tools documentation](https://pip-tools.readthedocs.io/en/stable/) · [Announcing Poetry 2.0.0](https://python-poetry.org/blog/announcing-poetry-2.0.0/) · [conda-forge/miniforge](https://github.com/conda-forge/miniforge) · [Anaconda Terms of Service](https://www.anaconda.com/legal/terms/terms-of-service)
- [python-dotenv](https://github.com/theskumar/python-dotenv) · [The Twelve-Factor App — Config](https://12factor.net/config)

### 🎥 Videos
- [Corey Schafer — Python Tutorial: UV - A Faster, All-in-One Package Manager to Replace Pip and Venv](https://www.youtube.com/watch?v=AMdG7IjgSPM) (~27 min) — calm, complete walkthrough of the uv project workflow shown in section 5
- [Corey Schafer — Python Tutorial: VENV (Mac & Linux) - How to Use Virtual Environments with the Built-In venv Module](https://www.youtube.com/watch?v=Kg1Yvry_Ydk) (~14 min) — the classic venv + pip basics, if section 2 went too fast
- [ArjanCodes — This Tool Replaces pip, Poetry, pyenv, and More (It’s Fast)](https://www.youtube.com/watch?v=qh98qOND6MI) (~18 min) — an experienced developer's view of switching a real workflow to uv

### 📄 Papers & Specifications
- [PEP 517 – A build-system independent format for source trees](https://peps.python.org/pep-0517/) · [PEP 621 – Storing project metadata in pyproject.toml](https://peps.python.org/pep-0621/)
- [PEP 420 – Implicit Namespace Packages](https://peps.python.org/pep-0420/) · [PEP 440 – Version Identification and Dependency Specification](https://peps.python.org/pep-0440/)
- [PEP 668 – Marking Python base environments as “externally managed”](https://peps.python.org/pep-0668/) · [PEP 735 – Dependency Groups in pyproject.toml](https://peps.python.org/pep-0735/) · [PEP 751 – A file format to record Python dependencies for installation reproducibility](https://peps.python.org/pep-0751/)

### 📘 Books & Courses
- [Welcome to Python Packages!](https://py-pkgs.org/) — the free *Python Packages* book by Tomas Beuzen & Tiffany Timbers
- [Scientific Python Library Development Guide](https://learn.scientific-python.org/development/) — modern best practices for research/ML libraries

### 🏋️ Practice
- [TestPyPI · The Python Package Index](https://test.pypi.org/) — publish the mini project safely
- [Entry points specification - Python Packaging User Guide](https://packaging.python.org/en/latest/specifications/entry-points/) — build a plugin system for your package as a follow-up exercise

## 📝 Summary Cheat Sheet

| Concept | What it does | Key API / command |
|---|---|---|
| Virtual environment | isolated interpreter + site-packages per project | `python -m venv .venv` · `uv venv` · `sys.prefix != sys.base_prefix` |
| pip | install packages into one interpreter | `python -m pip install "pkg>=1.0"` (quote!) |
| Version specifiers | declare acceptable versions | `==`, `>=,<`, `~=2.2`, `!=`, extras `[x]`, markers `; sys_platform == "linux"` |
| Lock file | exact transitive versions + hashes | `uv lock` · `uv.lock` · `pylock.toml` · `uv pip compile` |
| uv project | modern all-in-one workflow | `uv init --package` · `uv add` · `uv sync --locked` · `uv run` · `uvx` |
| Imports | find modules on `sys.path`, first match wins | regular package (`__init__.py`) · namespace package (PEP 420) · `python -m pkg` |
| pyproject.toml | build + metadata + tool config | `[build-system]` · `[project]` · `[project.scripts]` · `[dependency-groups]` · `[tool.*]` |
| Build | produce wheel + sdist | `python -m build` · `uv build` · backend `setuptools.build_meta` |
| Install | unpack wheel, create launchers | `pip install dist/*.whl` · editable: `pip install -e .` |
| Tests | verify the installed package | `pytest`, fixtures `tmp_path`/`capsys`, `parametrize`, `raises` |
| Lint/format | catch bugs, consistent style | `ruff check` · `ruff format` · `[tool.ruff.lint] select = [...]` |
| Reproducible ML | same env everywhere | pinned Python + lock + CUDA index markers + Docker + fingerprint |
| Secrets | keep keys out of code and git | env vars · `.env` (ignored) + `.env.example` · secret manager · rotate on leak |

## ➡️ What's Next

**[05 · Python Internals & Concurrency](05_Python_Internals_and_Concurrency.ipynb)** — with clean environments and packages in place, look under the hood: mutability and references, closures, the GIL, and when to use threads, processes, or `asyncio` for data loading and model-serving workloads.